# VIDA + PANDA: Multi-Agent VLM Crop Disease Diagnosis
### CDDM Dataset | 7 Agents | 4 Providers | ~25 imgs/class | Kaggle

**VIDA Block — Round 1 Tournament (all agents independent):**
- GPT-4.1 + GPT-4.1-mini (OpenAI)
- Claude Sonnet 4.5 + Claude Haiku 4.5 (Anthropic)
- Grok-4.20-Fast (xAI)
- Gemma-3n-E4B + Qwen3.5-9B (Together.ai, open-source)

**PANDA Block — Rounds 2+3 Named Debate (top-3 by R1 Combined Accuracy)**

**Judge:** GPT-5 (non-participant — removes the agent/judge self-bias flagged in review).
**Dataset name** is withheld from prompts (`MENTION_DATASET_NAME=False`) to rule out data leakage.

**Dataset:** CDDM — Add via Kaggle sidebar: `maljuboori/cddm-dataset`

---
## Kaggle Secrets Required
Add these in Settings -> Secrets:
- `OPENAI_API_KEY`
- `ANTHROPIC_API_KEY`
- `XAI_API_KEY`
- `TOGETHER_API_KEY`

## Workflow
1. Run setup cells (deps, keys, config, parser, agents, prompts)
2. Run **Agent Connectivity Test**, then **Qwen VLM Debug & Fallback** -> verify Qwen resolves
3. Run **Dataset Sampling** (now ~25 imgs/class)
4. Run **VIDA Round 1** (longer now — ~1,200 imgs x 7 agents)
5. Run **GPT-5 Judge**, **Metrics**, **Small-Sample Robustness**
6. Save R1 outputs as a Kaggle dataset, then use the standalone PANDA notebook
---


## Cell 1: Install Dependencies

In [ ]:
%%capture
!pip install openai anthropic together pillow pandas matplotlib seaborn tqdm scikit-learn requests -q
print("✅ Dependencies installed")

## Cell 2: Imports & API Keys

In [ ]:
import os, json, base64, random, time, re, math, shutil, traceback
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import requests as _requests
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import Counter, defaultdict

import openai
import anthropic
from together import Together

# ── Kaggle Secrets ────────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
_secrets = UserSecretsClient()

OPENAI_API_KEY    = _secrets.get_secret("OPENAI_API_KEY")
ANTHROPIC_API_KEY = _secrets.get_secret("ANTHROPIC_API_KEY")
XAI_API_KEY       = _secrets.get_secret("XAI_API_KEY")
TOGETHER_API_KEY  = _secrets.get_secret("TOGETHER_API_KEY")

# ── Initialize clients ────────────────────────────────────────────────────────
openai_client    = openai.OpenAI(api_key=OPENAI_API_KEY)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
xai_client       = openai.OpenAI(
    api_key=XAI_API_KEY,
    base_url="https://api.x.ai/v1",
)
together_client  = Together(api_key=TOGETHER_API_KEY)
_together_compat = openai.OpenAI(
    api_key=TOGETHER_API_KEY,
    base_url="https://api.together.xyz/v1",
    timeout=90.0,
    max_retries=0,
)

print("✅ OpenAI    :", openai_client.base_url)
print("✅ Anthropic : initialized")
print("✅ xAI       :", xai_client.base_url)
print("✅ Together  : initialized")

## Cell 3: Config & CDDM Taxonomy

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Auto-detect CDDM dataset path from Kaggle input
_cddm_candidates = [
    Path("/kaggle/input/cddm-dataset/images"),
    Path("/kaggle/input/datasets/maljuboori/cddm-dataset/images"),
]
DATASET_ROOT = None
for _c in _cddm_candidates:
    if _c.exists():
        DATASET_ROOT = _c
        break
if DATASET_ROOT is None:
    # Fallback: search recursively
    for p in Path("/kaggle/input").rglob("images"):
        if p.is_dir() and any(p.iterdir()):
            DATASET_ROOT = p
            break

OUTPUT_DIR = Path("/kaggle/working/results_cddm")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Run config ────────────────────────────────────────────────────────────────
MAX_PER_CLASS    = 25          # was effectively ~10 (~490 imgs); 25 -> ~1,200 imgs (reviewer R1.2 scale-up)
HEALTHY_FRACTION = 0.20        # keep ~20% healthy / ~80% diseased (same methodology, scaled)
RANDOM_SEED      = 42
SOFTMAX_TEMP     = 2.0
HEALTHY_LABEL    = "Healthy"
CHECKPOINT_EVERY = 10
N_IMAGES_R1      = None         # set by the sampling cell (actual pilot size)

# ── Judge / leakage / open-model config (review revisions) ────────────────────
JUDGE_MODEL          = "gpt-5"  # non-participant judge (reviewer R1.3 self-bias). Swappable.
MENTION_DATASET_NAME = False    # False = withhold 'CDDM' from prompts (reviewer R2-Meth1, anti-leakage)
QWEN_MODEL           = "Qwen/Qwen3.5-9B"   # SERVERLESS vision model, light+cheap (reviewer R2-Meth2 lighter setup)
QWEN_FALLBACKS       = [        # ONLY confirmed-serverless Qwen vision models (non-serverless ones 400)
    "Qwen/Qwen3.5-9B",
    "Qwen/Qwen3.5-397B-A17B",
]

random.seed(RANDOM_SEED)

# ── CDDM Taxonomy ─────────────────────────────────────────────────────────────
CDDM_CLASSES = [
    "Apple,Alternaria Blotch","Apple,Black Rot","Apple,Brown Spot",
    "Apple,Cedar Apple Rust","Apple,Frog Eye Leaf Spot","Apple,Grey Spot",
    "Apple,Healthy","Apple,Leaf Rust","Apple,Mosaic Virus",
    "Apple,Powdery Mildew","Apple,Scab",
    "Bell Pepper,Bacterial Spot","Bell Pepper,Healthy",
    "Blueberry,Healthy","Cherry,Healthy","Cherry,Powdery Mildew",
    "Corn,Healthy","Corn,Leaf Rust","Corn,Leaf Spot","Corn,Northern Leaf Blight",
    "Grape,Black Rot","Grape,Esca","Grape,Healthy","Grape,Leaf Blight",
    "Orange,Citrus Greening","Orange,Healthy",
    "Peach,Bacterial Spot","Peach,Healthy",
    "Potato,Early Blight","Potato,Healthy","Potato,Late Blight",
    "Pumpkin,Powdery Mildew","Raspberry,Healthy",
    "Rice,Bacterial Leaf Blight","Rice,Blast","Rice,Brown Spot",
    "Rice,Leaf Blight","Rice,Leaf Smut","Rice,Tungro",
    "Soybean,Healthy","Strawberry,Healthy","Strawberry,Leaf Scorch",
    "Tomato,Bacterial Spot","Tomato,Early Blight","Tomato,Healthy",
    "Tomato,Late Blight","Tomato,Leaf Mold","Tomato,Mosaic Virus",
    "Tomato,Powdery Mildew","Tomato,Septoria Leaf Spot","Tomato,Spider Mites",
    "Tomato,Target Spot","Tomato,Yellow Leaf Curl","Wheat,Healthy",
]
CDDM_CROPS    = sorted(set(c.split(",")[0].strip() for c in CDDM_CLASSES))
CDDM_DISEASES = sorted(set(c.split(",")[1].strip() for c in CDDM_CLASSES))
_CROP_LIST    = ", ".join(CDDM_CROPS)
_DISEASE_LIST = ", ".join(CDDM_DISEASES)

if DATASET_ROOT:
    n_dirs = sum(1 for d in DATASET_ROOT.iterdir() if d.is_dir())
    print(f"✅ Dataset: {DATASET_ROOT}")
    print(f"   Class folders found: {n_dirs}")
else:
    print("❌ CDDM dataset NOT found — add via Kaggle sidebar → Add Input")
    print("   Dataset: maljuboori/cddm-dataset")

print(f"✅ Taxonomy : {len(CDDM_CLASSES)} classes | {len(CDDM_CROPS)} crops | {len(CDDM_DISEASES)} diseases")
print(f"✅ Output   : {OUTPUT_DIR}")
print(f"✅ Sampling : up to {MAX_PER_CLASS} imgs/class (~{HEALTHY_FRACTION:.0%} healthy)")
print(f"✅ Judge    : {JUDGE_MODEL} | dataset-name in prompt: {MENTION_DATASET_NAME}")

## Cell 4: Response Parser (3-Pass Matching)

In [ ]:
def strip_md(text):
    text = re.sub(r"```[\w]*\n?", "", text)
    text = re.sub(r"\*{1,2}([^*]+)\*{1,2}", r"\1", text)
    return text.strip()

def _match_crop(t):
    t = t.strip().lower()
    # Pass 1+2: collect all matches from both directions, return longest
    all_m = set()
    for c in CDDM_CROPS:
        if c.lower() in t: all_m.add(c)
    for c in CDDM_CROPS:
        if t and t in c.lower(): all_m.add(c)
    if all_m: return sorted(all_m, key=len, reverse=True)[0]
    # Pass 3: word overlap
    t_words = set(t.split())
    best, best_score = None, 0
    for c in sorted(CDDM_CROPS, key=len, reverse=True):
        overlap = len(t_words & set(c.lower().split()))
        if overlap > best_score and (overlap >= 2 or overlap == len(t_words)):
            best, best_score = c, overlap
    return best

def _match_disease(t):
    t = t.strip().lower()
    all_m = set()
    for d in CDDM_DISEASES:
        if d.lower() in t: all_m.add(d)
    for d in CDDM_DISEASES:
        if t and t in d.lower(): all_m.add(d)
    if all_m: return sorted(all_m, key=len, reverse=True)[0]
    t_words = set(t.split())
    best, best_score, best_len = None, 0, 0
    for d in sorted(CDDM_DISEASES, key=len, reverse=True):
        overlap = len(t_words & set(d.lower().split()))
        if overlap >= 2 or (overlap >= 1 and overlap == len(t_words)):
            if overlap > best_score or (overlap == best_score and len(d) > best_len):
                best, best_score, best_len = d, overlap, len(d)
    return best

def parse_response(text, agent_name=""):
    result = {"crop": None, "disease": None, "diseased": None,
              "reasoning": "", "confidence": None,
              "influenced_by": None, "new_visual_evidence": None, "raw": text}
    if not text or str(text).startswith("ERROR"): return result
    clean = strip_md(str(text)); clean_l = clean.lower()
    lines = clean.split("\n"); in_rsn = False; rsn_lines = []
    known = ["crop category:", "disease category:", "diseased:", "reasoning:",
             "confidence:", "influenced by:", "new visual evidence:"]
    for line in lines:
        s = line.strip(); sl = s.lower()
        if any(sl.startswith(f) for f in known): in_rsn = False
        if sl.startswith("crop category:") or sl.startswith("crop:"):
            val = s.split(":", 1)[1].strip()
            result["crop"] = (next((c for c in CDDM_CROPS if c.lower() == val.lower()), None)
                              or _match_crop(val.lower()))
        elif sl.startswith("disease category:") or sl.startswith("disease:"):
            val = s.split(":", 1)[1].strip()
            result["disease"] = (next((d for d in CDDM_DISEASES if d.lower() == val.lower()), None)
                                 or _match_disease(val.lower()))
        elif sl.startswith("diseased:"):
            val = s.split(":", 1)[1].strip().lower()
            if re.search(r"\byes\b", val): result["diseased"] = 1
            elif re.search(r"\bno\b", val): result["diseased"] = 0
        elif sl.startswith("reasoning:"):
            result["reasoning"] = s.split(":", 1)[1].strip(); in_rsn = True
        elif in_rsn and s: rsn_lines.append(s)
        elif sl.startswith("confidence:"):
            nums = re.findall(r"[0-9]*\.?[0-9]+", s.split(":", 1)[1])
            if nums:
                c = float(nums[0]); result["confidence"] = c / 100 if c > 1 else c
        elif sl.startswith("influenced by:"):
            val = s.split(":", 1)[1].strip()
            result["influenced_by"] = "None" if val.lower() in (
                "none", "n/a", "na", "no one", "nobody", "") else val
        elif sl.startswith("new visual evidence:"):
            val = s.split(":", 1)[1].strip()
            result["new_visual_evidence"] = "NONE" if val.upper() in (
                "NONE", "N/A", "NA", "NOTHING", "") else val
    if rsn_lines: result["reasoning"] = (result["reasoning"] + " " + " ".join(rsn_lines)).strip()
    if result["crop"] is None: result["crop"] = _match_crop(clean_l)
    if result["disease"] is None: result["disease"] = _match_disease(clean_l)
    if result["diseased"] is None and result["disease"]:
        result["diseased"] = 0 if result["disease"] == HEALTHY_LABEL else 1
    if result["confidence"] is None:
        m = re.search(r"confidence[\s:]*([0-9]*\.?[0-9]+)", clean_l)
        if m:
            c = float(m.group(1)); result["confidence"] = c / 100 if c > 1 else c
    if result["crop"] is None and agent_name:
        print(f"   ⚠️  [{agent_name}] CROP parse FAILED: {repr(str(text)[:80])}")
    if result["disease"] is None and agent_name:
        print(f"   ⚠️  [{agent_name}] DISEASE parse FAILED: {repr(str(text)[:80])}")
    return result

print("✅ Parser defined (3-pass: exact match → reverse substring → word overlap)")

## Cell 5: Agent Inference Functions

In [ ]:
def encode_b64(path):
    with open(path, "rb") as f: return base64.b64encode(f.read()).decode()

def mime_type(path):
    return "image/jpeg" if str(path).lower().endswith(("jpg", "jpeg")) else "image/png"

# Dataset name withheld unless MENTION_DATASET_NAME=True (anti-leakage, reviewer R2-Meth1)
_DATASET_TAG = "from the CDDM crop disease dataset" if MENTION_DATASET_NAME else "from crop leaf photographs"
_SYSTEM_PROMPT = (
    f"You are an expert plant pathologist classifying crop leaf images {_DATASET_TAG}. "
    f"Valid crops (choose EXACTLY one): {_CROP_LIST}. "
    f"Valid diseases (choose EXACTLY one): {_DISEASE_LIST}. "
    "NEVER output a crop or disease not in these lists. "
    "If unsure, pick the closest match from the lists. "
    "Do NOT ask for clarification — always classify."
)

def _openai_call(client, model, image_path, prompt, retries=3, delay=5):
    b64 = encode_b64(image_path)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": _SYSTEM_PROMPT},
                    {"role": "user", "content": [
                        {"type": "image_url", "image_url": {
                            "url": f"data:{mime_type(image_path)};base64,{b64}",
                            "detail": "high"}},
                        {"type": "text", "text": prompt}
                    ]}
                ],
                max_tokens=600,
            )
            raw = (resp.choices[0].message.content or "").strip()
            if resp.choices[0].finish_reason not in ("stop", "end_turn"):
                print(f"   ⚠️  [{model}] finish={resp.choices[0].finish_reason}")
            return raw
        except Exception as e:
            if attempt < retries - 1:
                print(f"   ⚠️  [{model}] ({str(e)[:80]}), retry {attempt+1}...")
                time.sleep(delay)
            else: return f"ERROR: {e}"

def call_openai(model, image_path, prompt, retries=3, delay=5):
    return _openai_call(openai_client, model, image_path, prompt, retries, delay)

def call_xai(model, image_path, prompt, retries=3, delay=5):
    return _openai_call(xai_client, model, image_path, prompt, retries, delay)

def call_anthropic(model, image_path, prompt, retries=3, delay=5):
    b64 = encode_b64(image_path); mt = mime_type(image_path)
    for attempt in range(retries):
        try:
            resp = anthropic_client.messages.create(
                model=model, max_tokens=600,
                system=_SYSTEM_PROMPT,
                messages=[{"role": "user", "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": mt, "data": b64}},
                    {"type": "text", "text": prompt}
                ]}])
            return resp.content[0].text.strip()
        except Exception as e:
            if attempt < retries - 1:
                print(f"   ⚠️  [Anthropic/{model}] ({str(e)[:80]}), retry {attempt+1}...")
                time.sleep(delay)
            else: return f"ERROR: {e}"

# ── Together — URL upload (base64 rejected by large models) ───────────────────
_url_cache = {}

def _upload_to_url(image_path, retries=2):
    img_bytes = open(image_path, "rb").read()
    img_name  = Path(image_path).name
    img_mime  = mime_type(image_path)
    hosts = [
        ("catbox",    lambda: _requests.post(
            "https://catbox.moe/user/api.php",
            data={"reqtype": "fileupload"},
            files={"fileToUpload": (img_name, img_bytes, img_mime)}, timeout=30)),
        ("litterbox", lambda: _requests.post(
            "https://litterbox.catbox.moe/resources/internals/api.php",
            data={"reqtype": "fileupload", "time": "72h"},
            files={"fileToUpload": (img_name, img_bytes, img_mime)}, timeout=30)),
        ("tmpfiles",  lambda: _requests.post(
            "https://tmpfiles.org/api/v1/upload",
            files={"file": (img_name, img_bytes, img_mime)}, timeout=30)),
    ]
    for host_name, fn in hosts:
        for attempt in range(retries):
            try:
                r = fn()
                if r.status_code == 200:
                    url = (r.json().get("data", {}).get("url", "").replace(
                        "tmpfiles.org/", "tmpfiles.org/dl/")
                        if host_name == "tmpfiles" else r.text.strip())
                    if url.startswith("https://"):
                        return url
            except Exception: pass
            time.sleep(2)
    print(f"   ❌ All image hosts failed for {Path(image_path).name}")
    return None

def call_together(model, image_path, prompt, retries=3, delay=5):
    img_key = str(image_path)
    if img_key not in _url_cache:
        url = _upload_to_url(image_path)
        if url is None: return "ERROR: image upload failed"
        _url_cache[img_key] = url
    img_url = _url_cache[img_key]
    for attempt in range(retries):
        try:
            resp = _together_compat.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": _SYSTEM_PROMPT},
                    {"role": "user", "content": [
                        {"type": "image_url", "image_url": {"url": img_url}},
                        {"type": "text", "text": prompt},
                    ]}
                ],
                max_tokens=700, timeout=90.0,
                # Qwen3.5/3.6 think by default on Together -> disable, else empty content
                **({"extra_body": {"chat_template_kwargs": {"enable_thinking": False}}}
                   if ("qwen3.5" in model.lower() or "qwen3.6" in model.lower()) else {}),
            )
            raw = (resp.choices[0].message.content or "").strip()
            if not raw:
                if attempt < retries - 1:
                    _url_cache.pop(img_key, None)
                    new_url = _upload_to_url(image_path)
                    if new_url: _url_cache[img_key] = img_url = new_url
                    print(f"   ⚠️  [{model}] empty, retry {attempt+1} (fresh URL)...")
                    time.sleep(delay * 2); continue
                else: return f"ERROR: {model} empty after {retries} retries"
            return raw
        except Exception as e:
            err = str(e)
            if "corrupt" in err.lower() or "could not be processed" in err.lower():
                _url_cache.pop(img_key, None)
                new_url = _upload_to_url(image_path)
                if new_url: _url_cache[img_key] = img_url = new_url
            if attempt < retries - 1:
                print(f"   ⚠️  [Together/{model}] ({err[:80]}), retry {attempt+1} (fresh URL)...")
                time.sleep(delay)
            else: return f"ERROR: {err}"

# ── Agent Registry ────────────────────────────────────────────────────────────
AGENTS_R1 = {
    "GPT-4.1":       lambda img, p: call_openai("gpt-4.1", img, p),
    "GPT-4.1-mini":  lambda img, p: call_openai("gpt-4.1-mini", img, p),
    "Claude-Sonnet": lambda img, p: call_anthropic("claude-sonnet-4-5", img, p),
    "Claude-Haiku":  lambda img, p: call_anthropic("claude-haiku-4-5", img, p),
    "Grok-4.20-Fast":   lambda img, p: call_xai("grok-4.20-non-reasoning", img, p),
    "Gemma-3n-E4B":  lambda img, p: call_together("google/gemma-3n-E4B-it", img, p),
    "Qwen-VL":       lambda img, p: call_together(QWEN_MODEL, img, p),   # open-source (reviewer R2-Meth2)
}
AGENT_FN_MAP = dict(AGENTS_R1)

print(f"✅ {len(AGENTS_R1)} agents: {list(AGENTS_R1.keys())}")
print("   Providers: OpenAI | Anthropic | xAI | Together.ai")

## Cell 6: Prompts & Softmax Voting

In [ ]:
ROUND1_PROMPT = (
    "You are a plant science researcher analyzing a crop leaf image for disease diagnosis.\n"
    + ("This image is from the CDDM crop disease dataset.\n\n" if MENTION_DATASET_NAME else "\n")
    + "BEGIN YOUR RESPONSE WITH THESE FIELDS — NO TEXT BEFORE THEM:\n"
    + f"Crop Category: <one of: {_CROP_LIST}>\n"
    + f"Disease Category: <one of: {_DISEASE_LIST}>\n"
    + "Diseased: <Yes or No>\n"
    + "Reasoning: <2-3 sentences citing specific visual features>\n"
    + "Confidence: <0.0 to 1.0>"
)

_FMT_R2 = (
    "BEGIN YOUR RESPONSE WITH THESE FIELDS — NO TEXT BEFORE THEM:\n"
    f"Crop Category: <one of: {_CROP_LIST}>\n"
    f"Disease Category: <one of: {_DISEASE_LIST}>\n"
    "Diseased: <Yes or No>\n"
    "Reasoning: <address peers by name, cite specific visual features>\n"
    "Confidence: <0.0 to 1.0>\n"
    "Influenced By: <exact peer name who changed your verdict, or None>\n"
    "New Visual Evidence: <specific new visual feature you now observe, or NONE>"
)
_FMT_R3 = (
    "BEGIN YOUR RESPONSE WITH THESE FIELDS — NO TEXT BEFORE THEM:\n"
    f"Crop Category: <one of: {_CROP_LIST}>\n"
    f"Disease Category: <one of: {_DISEASE_LIST}>\n"
    "Diseased: <Yes or No>\n"
    "Reasoning: <final position, address peers by name>\n"
    "Confidence: <0.0 to 1.0>\n"
    "Influenced By: <exact peer name who changed your verdict vs R2, or None>\n"
    "New Visual Evidence: <specific new visual feature observed vs R2, or NONE>"
)

def make_r2_prompt(my_name, own_r1, others_r1):
    own = (f"YOUR Round 1 verdict:\n"
           f"  Crop: {own_r1.get('crop','?')} / Disease: {own_r1.get('disease','?')}\n"
           f"  Reasoning: {own_r1.get('reasoning','')[:120]}")
    peers = "".join(
        f"\n[{pn}]: {p.get('crop','?')} / {p.get('disease','?')}\n"
        f"  Reasoning: {p.get('reasoning','')[:120]}\n"
        for pn, p in others_r1.items()
    )
    return (f"You are {my_name}, a plant science researcher.\n\n{own}\n\n"
            f"Peers analyzed the SAME image:\n{peers}\n"
            f"Re-examine the image. Address {' and '.join(others_r1.keys())} by name. "
            f"AGREE or DISAGREE citing visual features. Change verdict ONLY if you observe "
            f"new visual evidence — not due to peer pressure alone.\n\n" + _FMT_R2)

def make_r3_prompt(my_name, own_r1, own_r2, others_r2):
    history = (f"YOUR history: R1={own_r1.get('crop','?')}/{own_r1.get('disease','?')} "
               f"-> R2={own_r2.get('crop','?')}/{own_r2.get('disease','?')}\n"
               f"  R2 Reasoning: {own_r2.get('reasoning','')[:120]}")
    peers = "".join(
        f"\n[{pn}] R2: {p.get('crop','?')}/{p.get('disease','?')}\n"
        f"  Reasoning: {p.get('reasoning','')[:120]}\n"
        for pn, p in others_r2.items()
    )
    return (f"You are {my_name}. FINAL ROUND.\n\n{history}\n\nPeers R2:\n{peers}\n"
            f"Commit to your FINAL verdict. Address {' and '.join(others_r2.keys())} by name.\n\n"
            + _FMT_R3)

def compute_agent_weights(agent_names, metrics_df):
    accs = np.array([float(metrics_df.loc[a, "Combined Acc"])
                     if a in metrics_df.index else 0.0 for a in agent_names])
    exp_a = np.exp(accs / SOFTMAX_TEMP); weights = exp_a / exp_a.sum()
    w_dict = dict(zip(agent_names, weights))
    print("   Voting weights (softmax T=2.0):")
    for n, w in sorted(w_dict.items(), key=lambda x: -x[1]):
        acc = float(metrics_df.loc[n, "Combined Acc"]) if n in metrics_df.index else 0.0
        print(f"      {n:<22}: acc={acc:.3f} -> weight={w:.3f}")
    return w_dict

def weighted_vote(values_dict, weights_dict):
    scores = {}
    for agent, val in values_dict.items():
        if val is None: continue
        scores[val] = scores.get(val, 0.0) + weights_dict.get(agent, 1.0 / max(len(values_dict), 1))
    if not scores: return None, {}, 0.0
    winner = max(scores, key=scores.get)
    return winner, scores, scores[winner]

def simple_vote(values):
    valid = [v for v in values if v is not None]
    if not valid: return None, {}, 0.0
    counts = Counter(valid); top = counts.most_common()
    if len(top) == 1 or top[0][1] > top[1][1]:
        w = top[0][0]; return w, dict(counts), counts[w] / len(valid)
    return sorted(top, key=lambda x: (-x[1], str(x[0])))[0][0], dict(counts), top[0][1] / len(valid)

def aggregate_votes(parsed_dict, weights_dict, round_label="r3"):
    agents = list(parsed_dict.keys())
    crop_w,  _, crop_aw = weighted_vote({a: parsed_dict[a]["crop"]     for a in agents}, weights_dict)
    dis_w,   _, dis_aw  = weighted_vote({a: parsed_dict[a]["disease"]  for a in agents}, weights_dict)
    dis_b,   _, _       = weighted_vote({a: parsed_dict[a]["diseased"] for a in agents}, weights_dict)
    crop_u,  _, crop_au = simple_vote([parsed_dict[a]["crop"]    for a in agents])
    dis_u,   _, dis_au  = simple_vote([parsed_dict[a]["disease"] for a in agents])
    confs = [parsed_dict[a]["confidence"] for a in agents if parsed_dict[a]["confidence"] is not None]
    maj = [a for a in agents if parsed_dict[a]["disease"] == dis_w]
    reasoning = " | ".join(parsed_dict[a]["reasoning"] for a in maj if parsed_dict[a]["reasoning"])
    pfx = f"{round_label}_"
    return {
        f"{pfx}consensus_crop":    crop_w,
        f"{pfx}consensus_disease": dis_w,
        f"{pfx}consensus_diseased": dis_b,
        f"{pfx}consensus_reasoning": reasoning,
        f"{pfx}mean_confidence":   float(np.mean(confs)) if confs else None,
        f"{pfx}crop_vote_agree_w": round(crop_aw, 3),
        f"{pfx}dis_vote_agree_w":  round(dis_aw, 3),
        f"{pfx}crop_vote_agree_u": round(crop_au, 3),
        f"{pfx}dis_vote_agree_u":  round(dis_au, 3),
        f"{pfx}individual_crops":    {a: parsed_dict[a]["crop"]    for a in agents},
        f"{pfx}individual_diseases": {a: parsed_dict[a]["disease"] for a in agents},
    }

def aggregate_debate(r1_p, r2_p, r3_p, weights_dict=None):
    if weights_dict is None: weights_dict = {a: 1.0 / len(r3_p) for a in r3_p}
    result = {}
    result.update(aggregate_votes(r3_p, weights_dict, round_label="r3"))
    result.update(aggregate_votes(r1_p, weights_dict, round_label="r1_baseline"))
    result["consensus_crop"]          = result["r3_consensus_crop"]
    result["consensus_disease"]       = result["r3_consensus_disease"]
    result["consensus_diseased"]      = result["r3_consensus_diseased"]
    result["consensus_reasoning"]     = result["r3_consensus_reasoning"]
    result["mean_confidence"]         = result["r3_mean_confidence"]
    result["crop_vote_agreement"]     = result["r3_crop_vote_agree_w"]
    result["disease_vote_agreement"]  = result["r3_dis_vote_agree_w"]
    result["individual_crops_r3"]     = result["r3_individual_crops"]
    result["individual_diseases_r3"]  = result["r3_individual_diseases"]
    return result

print("✅ Prompts, debate templates, and voting defined")

## Load VIDA R1 Results from Kaggle Dataset
**Skip the connectivity test, sampling, and R1 tournament — load the saved outputs instead.**

Upload your VIDA R1 outputs as a Kaggle dataset and set the path below:
-  — the full R1 predictions (all agents × all images)
-  — raw API responses keyed by image filename

This reconstructs , , , , and 
so all downstream cells (judge, metrics, dashboard, PANDA) work unchanged.


In [ ]:
# ── Path to your uploaded VIDA R1 Kaggle dataset ──────────────────────────────
R1_DATASET_DIR = Path("/kaggle/input/datasets/maljuboori/vida-r1-results-cddm-25pc")  # adjust if your slug differs

R1_CSV  = R1_DATASET_DIR / "r1_results (5).csv"
R1_JSON = R1_DATASET_DIR / "r1_raw_bank (5).json"

assert R1_CSV.exists(),  f"r1_results.csv not found at {R1_CSV} — check R1_DATASET_DIR"
assert R1_JSON.exists(), f"r1_raw_bank.json not found at {R1_JSON} — check R1_DATASET_DIR"

# ── Load R1 predictions ───────────────────────────────────────────────────────
r1_df = pd.read_csv(R1_CSV)
with open(R1_JSON) as f:
    r1_raw_bank = json.load(f)

# downstream judge cell iterates this list-of-dicts form (same row order as r1_df)
r1_results = r1_df.fillna({"reasoning": "", "pred_crop": "", "pred_disease": ""}).to_dict("records")

# ── Reconstruct variables that downstream cells expect ────────────────────────
agent_names  = list(r1_df["agent"].unique())
N_IMAGES_R1  = r1_df["image_name"].nunique()

# Reconstruct  (list of dicts) from the CSV — one entry per unique image
_img_meta = r1_df.drop_duplicates("image_name")[["image_name","class_label","gt_crop","gt_disease","gt_diseased"]]
pilot = []
for _, row in _img_meta.iterrows():
    # Try to resolve the actual image path on disk (for judge / viz cells that may need it)
    _img_path = DATASET_ROOT / row["class_label"] / row["image_name"] if DATASET_ROOT.exists() else None
    pilot.append({
        "image_path": _img_path,
        "image_name": row["image_name"],
        "class_label": row["class_label"],
        "gt_crop":     row["gt_crop"],
        "gt_disease":  row["gt_disease"],
        "gt_diseased": int(row["gt_diseased"]),
    })

print(f"✅ Loaded VIDA R1 results from {R1_DATASET_DIR}")
print(f"   Images : {N_IMAGES_R1:,}")
print(f"   Agents : {len(agent_names)} — {agent_names}")
print(f"   Rows   : {len(r1_df):,} (images × agents)")
print(f"   Classes: {r1_df['class_label'].nunique()}")
print(f"   Raw bank: {len(r1_raw_bank):,} images")


## Cell 10: GPT-5 Judge — Reasoning Quality & Plausibility

In [ ]:
# ── Robust judge wrapper (default GPT-5, a reasoning model) ───────────────────
# GPT-5 spends tokens on HIDDEN reasoning first. With a small budget it returns
# an EMPTY string (finish_reason='length'). So: use max_completion_tokens (NOT
# max_tokens), reasoning_effort='minimal', a generous budget, and grow-and-retry
# on empty output. Still works if you set JUDGE_MODEL back to 'gpt-4.1'.
def _judge_call(prompt, max_out=2000, retries=4):
    is_gpt5 = str(JUDGE_MODEL).startswith("gpt-5")
    for attempt in range(retries):
        try:
            kwargs = {"model": JUDGE_MODEL,
                      "messages": [{"role": "user", "content": prompt}]}
            if is_gpt5:
                kwargs["max_completion_tokens"] = max_out
                kwargs["reasoning_effort"] = "minimal"   # GPT-5 does NOT accept temperature
            else:
                kwargs["max_tokens"] = max_out
            resp = openai_client.chat.completions.create(**kwargs)
            txt = (resp.choices[0].message.content or "").strip()
            if not txt and resp.choices[0].finish_reason == "length":
                max_out = min(max_out * 2, 8000)   # reasoning ate the budget — grow & retry
                continue
            if txt:
                return txt
        except Exception:
            if attempt < retries - 1: time.sleep(3)
    return ""

def judge_reasoning(gt_crop, gt_disease, reasoning, pred_crop, pred_disease):
    if not reasoning: return {"reasoning_quality": None, "plausibility": None}
    prompt = (
        f"Ground truth: {gt_crop} with {gt_disease}\n"
        f"Agent predicted: {pred_crop} / {pred_disease}\n"
        f"Agent reasoning: {reasoning}\n\n"
        "Rate this plant disease diagnosis reasoning (1-10 each):\n"
        "Reasoning Quality: <1-10 — cites specific visual features?>\n"
        "Plausibility: <1-10 — scientifically plausible even if wrong?>\n"
        "Respond ONLY with those two lines."
    )
    text = _judge_call(prompt, max_out=2000)
    rq = re.search(r"Reasoning Quality:\s*([0-9.]+)", text)
    pl = re.search(r"Plausibility:\s*([0-9.]+)", text)
    return {"reasoning_quality": float(rq.group(1)) if rq else None,
            "plausibility":      float(pl.group(1)) if pl else None}

print(f"⏳ Judging {len(r1_results)} responses...")
judge_scores = [judge_reasoning(r["gt_crop"], r["gt_disease"], r["reasoning"],
                                r["pred_crop"], r["pred_disease"])
                for r in tqdm(r1_results, desc="Judging")]
r1_df["reasoning_quality"] = [s["reasoning_quality"] for s in judge_scores]
r1_df["plausibility"]      = [s["plausibility"]      for s in judge_scores]
r1_df.to_csv(OUTPUT_DIR / "r1_results.csv", index=False)
print("✅ Judge scores saved")

## Cell 11: Phase 1 Metrics

In [ ]:
def calibration_score(group):
    sub = group.dropna(subset=["confidence"])
    if len(sub) < 3: return None
    corr = sub["confidence"].corr(sub["both_correct"].astype(float))
    return round(corr, 3) if not np.isnan(corr) else None

metrics_rows = []
for agent in agent_names:
    g = r1_df[r1_df["agent"] == agent]
    metrics_rows.append({
        "Agent":             agent,
        "N":                 len(g),
        "Crop Acc":          round(g["crop_correct"].mean(), 3),
        "Disease Acc":       round(g["disease_correct"].mean(), 3),
        "Combined Acc":      round(g["both_correct"].mean(), 3),
        "Parse Fail %":      round(g["parse_failed"].mean() * 100, 1),
        "Mean Confidence":   round(g["confidence"].dropna().mean(), 3) if g["confidence"].notna().any() else None,
        "Conf Calibration":  calibration_score(g),
        "Reasoning Quality": round(g["reasoning_quality"].dropna().mean(), 2) if g["reasoning_quality"].notna().any() else None,
        "Plausibility":      round(g["plausibility"].dropna().mean(), 2)      if g["plausibility"].notna().any()      else None,
    })

metrics_df = pd.DataFrame(metrics_rows).set_index("Agent")
metrics_df = metrics_df.sort_values("Combined Acc", ascending=False)
metrics_df.to_csv(OUTPUT_DIR / "r1_metrics.csv")
print("📊 VIDA Rankings (sorted by Combined Acc):")
print(metrics_df.to_string())

## Cell 11b: Reviewer R1.2 — Small-Sample Robustness (N held constant)

In [ ]:
# Re-aggregate the SAME R1 predictions on a stratified ~490-image subsample so the
# ONLY thing changing vs the full run is N. Stable per-agent accuracy => the original
# 490-image numbers were not a small-sample artifact (directly answers reviewer R1.2).
SUBSAMPLE_N = 490
random.seed(RANDOM_SEED)

_meta = r1_df.drop_duplicates("image_name")[["image_name", "class_label"]]
_by_class = _meta.groupby("class_label")["image_name"].apply(list).to_dict()
_total = sum(len(v) for v in _by_class.values())

_sub = []
for cl, imgs in _by_class.items():
    k = max(1, round(len(imgs) * SUBSAMPLE_N / _total))
    _sub += random.sample(imgs, min(k, len(imgs)))
sub_imgs = set(_sub[:SUBSAMPLE_N])
_sub_df = r1_df[r1_df["image_name"].isin(sub_imgs)]

print(f"Full run   : {r1_df['image_name'].nunique():,} images")
print(f"Subsample  : {len(sub_imgs):,} images (stratified by class)\n")

rows = []
for agent in agent_names:
    gf = r1_df[r1_df["agent"] == agent]
    gs = _sub_df[_sub_df["agent"] == agent]
    full_acc = gf["both_correct"].mean()
    sub_acc  = gs["both_correct"].mean()
    rows.append({"Agent": agent,
                 "Combined (full)": round(full_acc, 3),
                 f"Combined (N={len(sub_imgs)})": round(sub_acc, 3),
                 "Delta": round(sub_acc - full_acc, 3)})

cmp_df = pd.DataFrame(rows).set_index("Agent").sort_values("Combined (full)", ascending=False)
cmp_df.to_csv(OUTPUT_DIR / "r1_subsample_robustness.csv")
print(cmp_df.to_string())
print("\nInterpretation: small per-agent Delta (within Wilson CI width) => the headline")
print("results are stable to sample size; the 490-image pilot is not a small-sample artifact.")


## Cell 12: VIDA Dashboard

In [ ]:
fig = plt.figure(figsize=(26, 20))
fig.suptitle("VIDA Block — VLM Tournament Dashboard (CDDM Crop Disease Diagnosis)",
             fontsize=15, fontweight="bold", y=1.01)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)
agents_ordered = metrics_df.index.tolist()
colors = plt.cm.tab10(np.linspace(0, 1, len(agents_ordered)))

def bar_chart(ax, values, title, ylabel, color_map=None, ylim=(0, 1.05)):
    x = np.arange(len(agents_ordered))
    bars = ax.bar(x, values, color=color_map if color_map else colors)
    ax.set_xticks(x)
    ax.set_xticklabels([a.replace("-", " ") for a in agents_ordered], rotation=30, ha="right", fontsize=8)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=9)
    if ylim: ax.set_ylim(ylim)
    ax.grid(axis="y", alpha=0.3)
    for bar, v in zip(bars, values):
        if v is not None and not np.isnan(float(v or 0)):
            ax.text(bar.get_x() + bar.get_width() / 2, float(v) + 0.01, f"{float(v):.2f}",
                    ha="center", fontsize=7, fontweight="bold")

bar_chart(fig.add_subplot(gs[0, 0]),
          [metrics_df.loc[a, "Crop Acc"] for a in agents_ordered], "Crop Identification Accuracy", "Accuracy")
bar_chart(fig.add_subplot(gs[0, 1]),
          [metrics_df.loc[a, "Disease Acc"] for a in agents_ordered], "Disease Identification Accuracy", "Accuracy")
bar_chart(fig.add_subplot(gs[0, 2]),
          [metrics_df.loc[a, "Combined Acc"] for a in agents_ordered], "Combined Accuracy", "Accuracy",
          color_map=["#2a9d8f" if i == 0 else "#e76f51" if i == len(agents_ordered)-1
                     else "#8ECAE6" for i in range(len(agents_ordered))])
pf = [metrics_df.loc[a, "Parse Fail %"] for a in agents_ordered]
bar_chart(fig.add_subplot(gs[1, 0]), pf, "Parse Failure Rate % (lower=better)", "%",
          color_map=["#e63946" if v > 20 else "#f4a261" if v > 5 else "#2a9d8f" for v in pf],
          ylim=(0, max(max(pf, default=0) * 1.2, 5)))
bar_chart(fig.add_subplot(gs[1, 1]),
          [metrics_df.loc[a, "Mean Confidence"] if pd.notna(metrics_df.loc[a, "Mean Confidence"]) else 0
           for a in agents_ordered], "Mean Confidence", "Confidence")
ax = fig.add_subplot(gs[1, 2])
cal = [metrics_df.loc[a, "Conf Calibration"] if pd.notna(metrics_df.loc[a, "Conf Calibration"]) else 0
       for a in agents_ordered]
bar_chart(ax, cal, "Confidence Calibration (corr. with correctness)", "Pearson r", ylim=(-1.1, 1.1))
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
bar_chart(fig.add_subplot(gs[2, 0]),
          [metrics_df.loc[a, "Reasoning Quality"] if pd.notna(metrics_df.loc[a, "Reasoning Quality"]) else 0
           for a in agents_ordered], "Reasoning Quality (GPT-5 Judge /10)", "Score", ylim=(0, 10.5))
bar_chart(fig.add_subplot(gs[2, 1]),
          [metrics_df.loc[a, "Plausibility"] if pd.notna(metrics_df.loc[a, "Plausibility"]) else 0
           for a in agents_ordered], "Plausibility (GPT-5 Judge /10)", "Score", ylim=(0, 10.5))
ax = fig.add_subplot(gs[2, 2])
heat_cols = ["Crop Acc", "Disease Acc", "Combined Acc", "Reasoning Quality", "Plausibility"]
heat = metrics_df[heat_cols].copy()
for col in heat_cols:
    mn, mx = heat[col].min(), heat[col].max()
    heat[col] = (heat[col] - mn) / (mx - mn + 1e-9)
sns.heatmap(heat, annot=metrics_df[heat_cols].round(2), fmt="", cmap="RdYlGn", ax=ax,
            xticklabels=[c.replace(" ", "\n") for c in heat_cols], yticklabels=agents_ordered)
ax.set_title("Normalized Performance Heatmap", fontsize=10, fontweight="bold")

plt.savefig(OUTPUT_DIR / "vida_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Dashboard saved")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# VIDA SAMPLE TEXT OUTPUT — 10 images, all 6 model responses per image
# Selection: 3 all-correct, 3 all-wrong, 2 split-vote, 2 random
# ══════════════════════════════════════════════════════════════════════════════

_agent_names = r1_df["agent"].unique().tolist()

def _n_correct(img_name):
    rows = r1_df[r1_df["image_name"] == img_name]
    return int(rows["both_correct"].sum())

def _all_correct(img_name):  return _n_correct(img_name) == len(_agent_names)
def _all_wrong(img_name):    return _n_correct(img_name) == 0
def _split_vote(img_name):   return 0 < _n_correct(img_name) < len(_agent_names)

unique_imgs = r1_df["image_name"].unique().tolist()
all_correct_s = [i for i in unique_imgs if _all_correct(i)][:3]
all_wrong_s   = [i for i in unique_imgs if _all_wrong(i)][:3]
split_s       = [i for i in unique_imgs if _split_vote(i)][:2]
used          = set(all_correct_s + all_wrong_s + split_s)
random_s      = [i for i in unique_imgs if i not in used][:2]
sample_imgs   = all_correct_s + all_wrong_s + split_s + random_s

SEP  = "=" * 72
THIN = "-" * 72

print(f"VIDA BLOCK — ROUND 1 SAMPLE OUTPUTS  ({len(sample_imgs)} images)")
print(f"Independent Analysis | No Peer Communication | {len(_agent_names)} Agents")
print()

for idx, img_name in enumerate(sample_imgs, 1):
    img_rows   = r1_df[r1_df["image_name"] == img_name]
    gt_crop    = img_rows["gt_crop"].values[0]
    gt_disease = img_rows["gt_disease"].values[0]
    n_correct  = _n_correct(img_name)

    if n_correct == len(_agent_names):   category = "ALL CORRECT"
    elif n_correct == 0:                 category = "ALL WRONG"
    else:                                category = f"SPLIT  {n_correct}/{len(_agent_names)} correct"

    print(SEP)
    print(f"  [{idx:02d}/{len(sample_imgs)}]  {category}")
    print(f"  Image   : {img_name}")
    print(f"  GT      : {gt_crop}  /  {gt_disease}")
    print(THIN)

    for agent in _agent_names:
        row = img_rows[img_rows["agent"] == agent]
        if len(row) == 0: continue
        r = row.iloc[0]
        pred_crop = str(r["pred_crop"])  if pd.notna(r["pred_crop"])  else "PARSE FAIL"
        pred_dis  = str(r["pred_disease"]) if pd.notna(r["pred_disease"]) else "PARSE FAIL"
        conf      = f"{r['confidence']:.2f}" if pd.notna(r["confidence"]) else "—"
        rq        = f"RQ={r['reasoning_quality']:.1f}/10" if pd.notna(r.get("reasoning_quality")) else ""
        crop_ok   = pred_crop == gt_crop
        dis_ok    = pred_dis  == gt_disease
        verdict   = "CORRECT" if (crop_ok and dis_ok) else ("crop-only" if crop_ok else "WRONG")
        reasoning = str(r["reasoning"])
        rsn_short = reasoning[:] 

        print(f"  [{verdict:10s}]  {agent}")
        print(f"    Crop    : {pred_crop}")
        print(f"    Disease : {pred_dis}")
        print(f"    Conf    : {conf}   {rq}")
        print(f"    Reason  : {rsn_short}")
        print()

    print()

print(SEP)
print(f"  Total images in VIDA : {len(unique_imgs)}")
print(f"  Sample shown         : {len(sample_imgs)}")
print(f"  All-correct images   : {len(all_correct_s)}")
print(f"  All-wrong images     : {len(all_wrong_s)}")
print(f"  Split-vote images    : {len(split_s)}")
print(SEP)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# VIDA SAMPLE VISUALIZATION — Research paper quality
# ══════════════════════════════════════════════════════════════════════════════

import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from PIL import Image as PILImage

_agent_names = r1_df["agent"].unique().tolist()
n_agents     = len(_agent_names)
n_imgs       = len(sample_imgs)

SHORT = {
    "GPT-4.1":        "GPT-4.1",
    "GPT-4.1-mini":   "GPT-4.1-mini",
    "Claude-Sonnet":  "Claude Sonnet",
    "Claude-Haiku":   "Claude Haiku",
    "Grok-4.20-Fast": "Grok-4.20",
    "Gemma-3n-E4B":   "Gemma-3n",
}

PROVIDER = {
    "GPT-4.1":        "OpenAI",
    "GPT-4.1-mini":   "OpenAI",
    "Claude-Sonnet":  "Anthropic",
    "Claude-Haiku":   "Anthropic",
    "Grok-4.20-Fast": "xAI",
    "Gemma-3n-E4B":   "Together",
}

COLORS = {
    "correct":    ("#1b5e20", "#e8f5e9", "#a5d6a7"),  # text, bg, header-bg
    "crop_only":  ("#bf360c", "#fff8e1", "#ffcc80"),
    "wrong":      ("#7f0000", "#ffebee", "#ef9a9a"),
    "parse_fail": ("#424242", "#f5f5f5", "#e0e0e0"),
}

def _colors(crop_c, dis_c, parse_fail):
    if parse_fail:       return COLORS["parse_fail"]
    if crop_c and dis_c: return COLORS["correct"]
    if crop_c:           return COLORS["crop_only"]
    return COLORS["wrong"]

# ── Figure: 2 sub-rows per image (label + content) ────────────────────────────
fig = plt.figure(figsize=(4.8 * (n_agents + 1), 5.2 * n_imgs))
fig.patch.set_facecolor("#ffffff")

outer = fig.add_gridspec(n_imgs, 1, hspace=0.14)

for row_i, img_name in enumerate(sample_imgs):
    img_rows   = r1_df[r1_df["image_name"] == img_name]
    gt_crop    = img_rows["gt_crop"].values[0]
    gt_disease = img_rows["gt_disease"].values[0]
    n_correct  = _n_correct(img_name)

    border_c = "#2e7d32" if n_correct == n_agents else                "#c62828" if n_correct == 0 else "#e65100"
    outcome  = "All agents correct" if n_correct == n_agents else                "All agents incorrect" if n_correct == 0 else                f"{n_correct} of {n_agents} agents correct"

    # 2-row split: thin label (7%) + content (93%)
    row_gs = outer[row_i].subgridspec(2, 1, hspace=0.0, height_ratios=[0.07, 0.93])
    inner  = row_gs[1].subgridspec(1, n_agents + 1, wspace=0.025,
                                    width_ratios=[1.6] + [1.0] * n_agents)

    # ── Label strip ───────────────────────────────────────────────────────────
    ax_lbl = fig.add_subplot(row_gs[0])
    ax_lbl.set_facecolor("#f0f0f0")
    ax_lbl.axis("off")
    ax_lbl.text(0.012, 0.5,
                f"Figure {row_i+1}.",
                ha="left", va="center", transform=ax_lbl.transAxes,
                fontsize=9, fontweight="bold", color="#333",
                fontfamily="DejaVu Sans")
    ax_lbl.text(0.065, 0.5,
                f"{outcome}   |   Ground truth: {gt_crop}  —  {gt_disease}",
                ha="left", va="center", transform=ax_lbl.transAxes,
                fontsize=9, color="#333", fontfamily="DejaVu Sans")
    ax_lbl.axhline(0, color=border_c, linewidth=2.5)

    # ── Image panel ───────────────────────────────────────────────────────────
    ax_img = fig.add_subplot(inner[0])
    img_path = next((p["image_path"] for p in pilot
                     if p["image_path"].name == img_name), None)
    if img_path and Path(img_path).exists():
        ax_img.imshow(np.array(PILImage.open(img_path).convert("RGB")), aspect="auto")
    else:
        ax_img.set_facecolor("#f5f5f5")
        ax_img.text(0.5, 0.5, "Image unavailable", ha="center", va="center",
                    transform=ax_img.transAxes, fontsize=9, color="#bbb")
    ax_img.set_xticks([]); ax_img.set_yticks([])
    for sp in ax_img.spines.values():
        sp.set_edgecolor(border_c); sp.set_linewidth(2.5)

    # ── Agent cards ───────────────────────────────────────────────────────────
    for col_i, agent in enumerate(_agent_names):
        ax = fig.add_subplot(inner[col_i + 1])
        row_data = img_rows[img_rows["agent"] == agent]
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)

        if len(row_data) == 0:
            ax.set_facecolor("#f5f5f5")
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8, color="#bbb")
            continue

        r = row_data.iloc[0]
        pred_crop  = str(r["pred_crop"])    if pd.notna(r["pred_crop"])    else None
        pred_dis   = str(r["pred_disease"]) if pd.notna(r["pred_disease"]) else None
        crop_c     = int(pred_crop == gt_crop)     if pred_crop else 0
        dis_c      = int(pred_dis  == gt_disease)  if pred_dis  else 0
        parse_fail = bool(r["parse_failed"])
        conf       = f"{r['confidence']:.2f}" if pd.notna(r["confidence"]) else "N/A"
        rq_val     = r.get("reasoning_quality")
        rq         = f"{rq_val:.1f}" if pd.notna(rq_val) else "—"
        reasoning  = str(r["reasoning"]) if pd.notna(r["reasoning"]) else ""

        txt_c, bg_c, hdr_c = _colors(crop_c, dis_c, parse_fail)
        ax.set_facecolor(bg_c)
        for sp in ax.spines.values():
            sp.set_edgecolor(txt_c); sp.set_linewidth(1.2)

        verdict_str = "Correct"        if (crop_c and dis_c) else                       "Crop match"     if crop_c else                       "Parse failure"  if parse_fail else "Incorrect"

        # ── Header band ───────────────────────────────────────────────────────
        ax.add_patch(plt.Rectangle((0, 0.83), 1, 0.17, transform=ax.transAxes,
                                    facecolor=hdr_c, zorder=0))
        ax.axhline(0.83, color=txt_c, linewidth=0.8, alpha=0.5)

        # Agent name
        name_str = SHORT.get(agent, agent)
        ax.text(0.5, 0.975, name_str,
                ha="center", va="top", transform=ax.transAxes,
                fontsize=8.5, fontweight="bold", color=txt_c)
        # Provider tag
        prov = PROVIDER.get(agent, "")
        ax.text(0.5, 0.895, prov,
                ha="center", va="top", transform=ax.transAxes,
                fontsize=6.5, color=txt_c, alpha=0.75,
                style="italic")

        # ── Verdict badge ─────────────────────────────────────────────────────
        ax.text(0.5, 0.78, verdict_str,
                ha="center", va="top", transform=ax.transAxes,
                fontsize=8, fontweight="bold", color=txt_c)

        ax.axhline(0.72, color=txt_c, linewidth=0.5, alpha=0.25)

        # ── Prediction fields ─────────────────────────────────────────────────
        # Crop
        ax.text(0.06, 0.685, "Crop", ha="left", va="top",
                transform=ax.transAxes, fontsize=6.5, color="#888",
                fontweight="bold", style="italic")
        crop_txt = (pred_crop or "—")
        # Truncate long names gracefully
        if len(crop_txt) > 20: crop_txt = crop_txt[:19] + "."
        ax.text(0.06, 0.625, crop_txt, ha="left", va="top",
                transform=ax.transAxes, fontsize=8, color="#111")

        # Disease
        ax.text(0.06, 0.560, "Disease", ha="left", va="top",
                transform=ax.transAxes, fontsize=6.5, color="#888",
                fontweight="bold", style="italic")
        dis_txt = (pred_dis or "—")
        # Wrap long disease names
        if len(dis_txt) > 22:
            mid = dis_txt.rfind(" ", 0, 22)
            dis_txt = dis_txt[:mid] + "\n" + dis_txt[mid+1:] if mid > 0 else dis_txt[:22] + "."
        ax.text(0.06, 0.500, dis_txt, ha="left", va="top",
                transform=ax.transAxes, fontsize=7.5, color="#111",
                linespacing=1.3)

        ax.axhline(0.42, color="#ccc", linewidth=0.5)

        # ── Metrics row ───────────────────────────────────────────────────────
        ax.text(0.06, 0.390, f"Confidence: {conf}",
                ha="left", va="top", transform=ax.transAxes,
                fontsize=7, color="#444")
        ax.text(0.06, 0.330, f"Reasoning Quality: {rq}/10",
                ha="left", va="top", transform=ax.transAxes,
                fontsize=7, color="#444")

        ax.axhline(0.285, color="#ccc", linewidth=0.5)

        # ── Reasoning excerpt ─────────────────────────────────────────────────
        ax.text(0.06, 0.265, "Reasoning excerpt",
                ha="left", va="top", transform=ax.transAxes,
                fontsize=6, color="#999", style="italic")

        words = reasoning.split()
        lines_out, line = [], []
        for w in words:
            line.append(w)
            if len(" ".join(line)) > 34:
                lines_out.append(" ".join(line[:-1]))
                line = [w]
            if len(lines_out) == 3: break
        if line and len(lines_out) < 3: lines_out.append(" ".join(line))
        excerpt = "\n".join(lines_out[:3])
        if len(words) > len(excerpt.split()): excerpt += "..."

        ax.text(0.06, 0.225, excerpt,
                ha="left", va="top", transform=ax.transAxes,
                fontsize=6.5, color="#555", style="italic",
                linespacing=1.4)

        # Column header (first row only)
        if row_i == 0:
            ax.set_title(SHORT.get(agent, agent), fontsize=9.5,
                         fontweight="bold", pad=5, color="#222",
                         fontfamily="DejaVu Sans")

# ── Figure title & legend ─────────────────────────────────────────────────────
fig.suptitle(
    "VIDA Block — Round 1 Independent Analysis  |  CDDM Crop Disease Diagnosis Dataset",
    fontsize=13, fontweight="bold", y=1.004, color="#111",
    fontfamily="DejaVu Sans"
)

legend_patches = [
    mpatches.Patch(facecolor="#e8f5e9", edgecolor="#1b5e20", label="Correct (crop + disease)"),
    mpatches.Patch(facecolor="#fff8e1", edgecolor="#bf360c", label="Crop match only"),
    mpatches.Patch(facecolor="#ffebee", edgecolor="#7f0000", label="Incorrect"),
    mpatches.Patch(facecolor="#f5f5f5", edgecolor="#424242", label="Parse failure"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=4,
           fontsize=9, bbox_to_anchor=(0.5, -0.006),
           framealpha=1.0, edgecolor="#ccc", frameon=True)

plt.savefig(OUTPUT_DIR / "vida_sample_outputs.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.show()
print("[OK] VIDA sample visualization saved: vida_sample_outputs.png")


## Cell 13: Agent Selection for PANDA

Review the dashboard. Set `AUTO_SELECT=True` for automatic top-3,
or `AUTO_SELECT=False` and fill `MANUAL_AGENTS`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
AUTO_SELECT   = True    # True = auto top-N | False = manual
N_AUTO        = 3       # number of debate agents

MANUAL_AGENTS = [       # only used when AUTO_SELECT = False
    "GPT-4.1",
    "Grok-3",
    "Claude-Sonnet",
]
# ══════════════════════════════════════════════════════════════════════════════

if AUTO_SELECT:
    DEBATE_AGENTS = metrics_df.sort_values("Combined Acc", ascending=False).head(N_AUTO).index.tolist()
    print(f"✅ AUTO-SELECTED top-{N_AUTO}: {DEBATE_AGENTS}")
else:
    invalid = [a for a in MANUAL_AGENTS if a not in agent_names]
    if invalid: raise ValueError(f"Invalid agents: {invalid}. Valid: {agent_names}")
    DEBATE_AGENTS = MANUAL_AGENTS
    print(f"✅ MANUAL selection: {DEBATE_AGENTS}")

print(f"\n📊 Selected agents R1 stats:")
print(metrics_df.loc[DEBATE_AGENTS, ["Crop Acc", "Disease Acc", "Combined Acc", "Reasoning Quality"]].to_string())
AGENT_WEIGHTS = compute_agent_weights(DEBATE_AGENTS, metrics_df)
AGENTS_DEBATE = {name: AGENT_FN_MAP[name] for name in DEBATE_AGENTS}
print(f"\n✅ PANDA ready. Weights computed.")

## Cell 14: Pre-PANDA Debate Test
Run this before the full PANDA run. Send the full output before proceeding.

In [ ]:
# Find a test image where agents disagreed in R1
_test_sample = None
for sample in pilot[:50]:
    img_name = sample["image_path"].name
    if img_name not in r1_raw_bank: continue
    r1_for_img = {n: parse_response(r1_raw_bank[img_name].get(n, "")) for n in DEBATE_AGENTS}
    unique = len(set(r1_for_img[n]["crop"] for n in DEBATE_AGENTS if r1_for_img[n]["crop"]))
    if unique >= 2:
        _test_sample = sample; break
if _test_sample is None:
    _test_sample = next(s for s in pilot if s["image_path"].name in r1_raw_bank)

img_path   = _test_sample["image_path"]
img_name   = img_path.name
gt_crop    = _test_sample["gt_crop"]
gt_disease = _test_sample["gt_disease"]

print("=" * 70)
print("  PRE-PANDA DEBATE TEST")
print("=" * 70)
print(f"  Image  : {img_name}")
print(f"  GT     : {gt_crop} / {gt_disease}")
print(f"  Agents : {DEBATE_AGENTS}\n")

r1_p = {n: parse_response(r1_raw_bank[img_name].get(n, "")) for n in DEBATE_AGENTS}
print("  R1 (pre-debate):")
for n in DEBATE_AGENTS:
    p = r1_p[n]
    ok = "✅" if (p["crop"]==gt_crop and p["disease"]==gt_disease) else ("🌿" if p["crop"]==gt_crop else "❌")
    print(f"    {ok} {n:<22}: {p['crop']}/{p['disease']} (conf={p['confidence']})")

print("\n  R2 (named debate):")
r2_raw = {}; r2_p = {}
for n, fn in AGENTS_DEBATE.items():
    others = {k: v for k, v in r1_p.items() if k != n}
    raw = fn(img_path, make_r2_prompt(n, r1_p[n], others))
    r2_raw[n] = raw; r2_p[n] = parse_response(raw, agent_name=f"TEST-R2-{n}")
    p = r2_p[n]
    ok = "✅" if (p["crop"]==gt_crop and p["disease"]==gt_disease) else ("🌿" if p["crop"]==gt_crop else "❌")
    inf = p["influenced_by"] or "None"; ev = (p["new_visual_evidence"] or "NONE")[:35]
    changed = " 🔄 CHANGED" if (r1_p[n]["disease"] != p["disease"] and r1_p[n]["disease"] and p["disease"]) else ""
    print(f"    {ok} {n:<22}: {p['crop']}/{p['disease']} | inf={inf} | ev={ev}{changed}")
    print(f"       {(p['reasoning'] or '')[:100]}")

print("\n  R3 (final verdict):")
r3_raw = {}; r3_p = {}
for n, fn in AGENTS_DEBATE.items():
    others = {k: v for k, v in r2_p.items() if k != n}
    raw = fn(img_path, make_r3_prompt(n, r1_p[n], r2_p[n], others))
    r3_raw[n] = raw; r3_p[n] = parse_response(raw, agent_name=f"TEST-R3-{n}")
    p = r3_p[n]
    ok = "✅" if (p["crop"]==gt_crop and p["disease"]==gt_disease) else ("🌿" if p["crop"]==gt_crop else "❌")
    print(f"    {ok} {n:<22}: {p['crop']}/{p['disease']} conf={p['confidence']} inf={p['influenced_by']}")

cons = aggregate_debate(r1_p, r2_p, r3_p, weights_dict=AGENT_WEIGHTS)
b_ok = int(cons.get("r1_baseline_consensus_crop","")==gt_crop and cons.get("r1_baseline_consensus_disease","")==gt_disease)
p_ok = int(cons["consensus_crop"]==gt_crop and cons["consensus_disease"]==gt_disease)
print(f"\n  PRE-DEBATE  consensus: {cons.get('r1_baseline_consensus_crop','?')}/{cons.get('r1_baseline_consensus_disease','?')} {'✅' if b_ok else '❌'}")
print(f"  POST-DEBATE consensus: {cons['consensus_crop']}/{cons['consensus_disease']} {'✅' if p_ok else '❌'}")
print("\n  ✅ Send this output before running Cell 15 (full PANDA)")
print("=" * 70)

## Cell 15: PANDA Block — Run R2 + R3 Debate

In [ ]:
# ── Resume from checkpoint ────────────────────────────────────────────────────
_debate_ckpt = OUTPUT_DIR / "debate_results_checkpoint.csv"
_trans_ckpt  = OUTPUT_DIR / "debate_transcripts_checkpoint.json"

if _debate_ckpt.exists():
    _ex = pd.read_csv(_debate_ckpt)
    _done_debate   = set(_ex["image_name"].tolist())
    debate_results = _ex.to_dict("records")
    print(f"♻️  Resuming debate: {len(_done_debate)} done")
else:
    _done_debate = set(); debate_results = []

debate_transcripts = json.load(open(_trans_ckpt)) if _trans_ckpt.exists() else {}
debate_agents_list = list(AGENTS_DEBATE.keys())
_remaining_debate  = [s for s in pilot if s["image_path"].name not in _done_debate]

print(f"🚀 PANDA: {len(_remaining_debate)} remaining × {len(debate_agents_list)} agents × 2 rounds")
print(f"   Agents: {debate_agents_list}\n")

for i, sample in enumerate(tqdm(_remaining_debate, desc="PANDA")):
    img_path    = sample["image_path"]
    img_name    = img_path.name
    gt_crop     = sample["gt_crop"]
    gt_disease  = sample["gt_disease"]
    gt_diseased = sample["gt_diseased"]

    print(f"\n[{len(_done_debate)+i+1}/{len(pilot)}] {img_name} | GT: {gt_crop}/{gt_disease}")

    r1_p = {n: parse_response(r1_raw_bank.get(img_name, {}).get(n, "")) for n in debate_agents_list}
    result = {"image_name": img_name, "class_label": sample["class_label"],
              "gt_crop": gt_crop, "gt_disease": gt_disease, "gt_diseased": gt_diseased}
    for n in debate_agents_list:
        result[f"r1_{n}_crop"] = r1_p[n]["crop"]; result[f"r1_{n}_disease"] = r1_p[n]["disease"]

    print("  [R2] Named debate...")
    r2_raw = {}; r2_p = {}
    for n, fn in AGENTS_DEBATE.items():
        others = {k: v for k, v in r1_p.items() if k != n}
        raw = fn(img_path, make_r2_prompt(n, r1_p[n], others))
        r2_raw[n] = raw; r2_p[n] = parse_response(raw, agent_name=f"{n} R2")
        result[f"r2_{n}_crop"]            = r2_p[n]["crop"]
        result[f"r2_{n}_disease"]         = r2_p[n]["disease"]
        result[f"r2_{n}_influenced_by"]   = r2_p[n]["influenced_by"]
        result[f"r2_{n}_new_evidence"]    = r2_p[n]["new_visual_evidence"]
        result[f"r2_{n}_crop_changed"]    = int(bool(r1_p[n]["crop"]) and bool(r2_p[n]["crop"]) and r1_p[n]["crop"] != r2_p[n]["crop"])
        result[f"r2_{n}_disease_changed"] = int(bool(r1_p[n]["disease"]) and bool(r2_p[n]["disease"]) and r1_p[n]["disease"] != r2_p[n]["disease"])
        print(f"    {n}: {r2_p[n]['crop']}/{r2_p[n]['disease']} | inf={r2_p[n]['influenced_by']}")

    print("  [R3] Final verdict...")
    r3_raw = {}; r3_p = {}
    for n, fn in AGENTS_DEBATE.items():
        others = {k: v for k, v in r2_p.items() if k != n}
        raw = fn(img_path, make_r3_prompt(n, r1_p[n], r2_p[n], others))
        r3_raw[n] = raw; r3_p[n] = parse_response(raw, agent_name=f"{n} R3")
        result[f"r3_{n}_crop"]            = r3_p[n]["crop"]
        result[f"r3_{n}_disease"]         = r3_p[n]["disease"]
        result[f"r3_{n}_confidence"]      = r3_p[n]["confidence"]
        result[f"r3_{n}_influenced_by"]   = r3_p[n]["influenced_by"]
        result[f"r3_{n}_new_evidence"]    = r3_p[n]["new_visual_evidence"]
        result[f"r3_{n}_crop_changed"]    = int(bool(r2_p[n]["crop"]) and bool(r3_p[n]["crop"]) and r2_p[n]["crop"] != r3_p[n]["crop"])
        result[f"r3_{n}_disease_changed"] = int(bool(r2_p[n]["disease"]) and bool(r3_p[n]["disease"]) and r2_p[n]["disease"] != r3_p[n]["disease"])
        print(f"    {n}: {r3_p[n]['crop']}/{r3_p[n]['disease']} conf={r3_p[n]['confidence']} inf={r3_p[n]['influenced_by']}")

    cons = aggregate_debate(r1_p, r2_p, r3_p, weights_dict=AGENT_WEIGHTS)
    result.update(cons)
    result["consensus_crop_correct"]    = int(cons["consensus_crop"] == gt_crop)    if cons["consensus_crop"]    else 0
    result["consensus_disease_correct"] = int(cons["consensus_disease"] == gt_disease) if cons["consensus_disease"] else 0
    result["consensus_both_correct"]    = int(result["consensus_crop_correct"] and result["consensus_disease_correct"])
    result["baseline_crop_correct"]     = int(cons.get("r1_baseline_consensus_crop", "") == gt_crop)
    result["baseline_disease_correct"]  = int(cons.get("r1_baseline_consensus_disease", "") == gt_disease)
    result["baseline_both_correct"]     = int(result["baseline_crop_correct"] and result["baseline_disease_correct"])
    for n in debate_agents_list:
        for rnd, p in [("r1", r1_p), ("r2", r2_p), ("r3", r3_p)]:
            result[f"{rnd}_{n}_crop_correct"]    = int(p[n]["crop"] == gt_crop)     if p[n]["crop"]    else 0
            result[f"{rnd}_{n}_disease_correct"] = int(p[n]["disease"] == gt_disease) if p[n]["disease"] else 0
            result[f"{rnd}_{n}_both_correct"]    = int(result[f"{rnd}_{n}_crop_correct"] and result[f"{rnd}_{n}_disease_correct"])

    debate_results.append(result)
    debate_transcripts[img_name] = {"r2": r2_raw, "r3": r3_raw}

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(debate_results).to_csv(_debate_ckpt, index=False)
        with open(_trans_ckpt, "w") as f: json.dump(debate_transcripts, f)
        print(f"   💾 Checkpoint ({len(_done_debate)+i+1} images)")

debate_df = pd.DataFrame(debate_results)
debate_df.to_csv(OUTPUT_DIR / "debate_results.csv", index=False)
with open(OUTPUT_DIR / "debate_transcripts.json", "w") as f: json.dump(debate_transcripts, f, indent=2)
print(f"\n✅ PANDA complete — {len(debate_results)} rows")

## Cell 16: Influence & Convergence Analysis

In [ ]:
influence_matrix = defaultdict(int)
stubbornness  = {n: 0 for n in debate_agents_list}
flip_flop     = {n: 0 for n in debate_agents_list}
change_helped = {n: 0 for n in debate_agents_list}
change_hurt   = {n: 0 for n in debate_agents_list}
evidence_driven = {n: 0 for n in debate_agents_list}
n_img = len(debate_df)

for _, row in debate_df.iterrows():
    for n in debate_agents_list:
        ch_r2 = row.get(f"r2_{n}_disease_changed", 0)
        ch_r3 = row.get(f"r3_{n}_disease_changed", 0)
        ev_r2 = row.get(f"r2_{n}_new_evidence", "NONE")
        if not ch_r2 and not ch_r3: stubbornness[n] += 1
        if ch_r2 and ch_r3:         flip_flop[n] += 1
        if ch_r2 and ev_r2 and ev_r2 != "NONE": evidence_driven[n] += 1
        if ch_r2:
            r1c = int(row.get(f"r1_{n}_disease", "") == row["gt_disease"])
            r2c = int(row.get(f"r2_{n}_disease", "") == row["gt_disease"])
            if not r1c and r2c: change_helped[n] += 1
            elif r1c and not r2c: change_hurt[n] += 1
        for rnd in ["r2", "r3"]:
            inf = row.get(f"{rnd}_{n}_influenced_by")
            if inf and inf != "None" and pd.notna(inf):
                for cand in debate_agents_list:
                    if cand.lower() in str(inf).lower():
                        influence_matrix[(cand, n)] += 1; break

print("📊 Influence & Convergence:")
print(f"  {'Agent':<22} {'Stubborn%':>10} {'Flip%':>8} {'EvDriven':>9} {'Helped':>7} {'Hurt':>6}")
for n in debate_agents_list:
    print(f"  {n:<22} {stubbornness[n]/n_img:>10.1%} {flip_flop[n]/n_img:>8.1%} "
          f"{evidence_driven[n]:>9} {change_helped[n]:>7} {change_hurt[n]:>6}")
print(f"\n  Influence Matrix (row=persuader, col=changed):")
print(f"  {'':22}", end="")
for n in debate_agents_list: print(f" {n:>14}", end="")
print()
for inf_n in debate_agents_list:
    print(f"  {inf_n:<22}", end="")
    for chg_n in debate_agents_list:
        v = "—" if inf_n == chg_n else str(influence_matrix.get((inf_n, chg_n), 0))
        print(f" {v:>14}", end="")
    print()
pd.DataFrame([{"influencer": k[0], "changed": k[1], "count": v}
              for k, v in influence_matrix.items()]).to_csv(
    OUTPUT_DIR / "influence_matrix.csv", index=False)
print("\n✅ Influence matrix saved")

## Cell 17: Pre-Debate vs Post-Debate Accuracy

In [ ]:
base_crop = debate_df["baseline_crop_correct"].mean()
base_dis  = debate_df["baseline_disease_correct"].mean()
base_both = debate_df["baseline_both_correct"].mean()
cons_crop = debate_df["consensus_crop_correct"].mean()
cons_dis  = debate_df["consensus_disease_correct"].mean()
cons_both = debate_df["consensus_both_correct"].mean()
delta     = cons_both - base_both

print("=" * 70)
print("  PANDA IMPACT: Pre-Debate Baseline vs Post-Debate Consensus")
print("=" * 70)
print(f"  {'':28} {'Crop':>8} {'Disease':>9} {'Combined':>10}")
print("  " + "-" * 58)
print(f"  {'Pre-Debate (R1 weighted):':<28} {base_crop:>8.3f} {base_dis:>9.3f} {base_both:>10.3f}")
arr = "[^]" if delta > 0.01 else ("[v]" if delta < -0.01 else "[=]")
print(f"  {'Post-Debate (R3 weighted):':<28} {cons_crop:>8.3f} {cons_dis:>9.3f} {cons_both:>10.3f}  {arr} d={delta:+.3f}")
print()
if delta > 0.01:   print(f"  ++ Debate IMPROVED combined accuracy by {delta:.1%}")
elif delta < -0.01: print(f"  -- Debate HURT combined accuracy by {abs(delta):.1%}")
else:               print(f"  == Debate had no significant effect")
print()

print("  PER-AGENT:")
print(f"  {'Agent':<22} {'R1 Crop':>8} {'R1 Dis':>7} {'R1 Both':>8} | {'R3 Crop':>8} {'R3 Dis':>8} {'R3 Both':>8} {'Weight':>8}")
print("  " + "-" * 75)
for n in debate_agents_list:
    r1c = debate_df.apply(lambda r: int(r.get(f"r1_{n}_crop", "") == r["gt_crop"]), axis=1).mean()
    r1d = debate_df.apply(lambda r: int(r.get(f"r1_{n}_disease", "") == r["gt_disease"]), axis=1).mean()
    r1b = debate_df.apply(lambda r: int(r.get(f"r1_{n}_crop", "") == r["gt_crop"] and
                                        r.get(f"r1_{n}_disease", "") == r["gt_disease"]), axis=1).mean()
    r3c = debate_df.apply(lambda r: int(r.get(f"r3_{n}_crop", "") == r["gt_crop"]), axis=1).mean()
    r3d = debate_df.apply(lambda r: int(r.get(f"r3_{n}_disease", "") == r["gt_disease"]), axis=1).mean()
    r3b = debate_df.apply(lambda r: int(r.get(f"r3_{n}_crop", "") == r["gt_crop"] and
                                        r.get(f"r3_{n}_disease", "") == r["gt_disease"]), axis=1).mean()
    da = r3b - r1b; arr = "^" if da > 0.01 else ("v" if da < -0.01 else "=")
    wt = AGENT_WEIGHTS.get(n, 0)
    print(f"  {n:<22} {r1c:>8.3f} {r1d:>7.3f} {r1b:>8.3f} | {r3c:>8.3f} {r3d:>8.3f} {r3b:>8.3f} {arr} w={wt:.3f}")
print(f"  {'CONSENSUS':<22} {'':>8} {'':>7} {'':>8} | {cons_crop:>8.3f} {cons_dis:>8.3f} {cons_both:>8.3f}")
print("=" * 70)

## Cell 18: PANDA Visualizations

In [ ]:
# ── 6-panel analysis ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
fig.suptitle(f"PANDA Block — {', '.join(debate_agents_list)}",
             fontsize=13, fontweight="bold", y=1.01)
x = np.arange(len(debate_agents_list)); w = 0.3
colors_rnd = ["#8ECAE6", "#023047"]

def _per(df, agents, rnd, metric):
    return [df.apply(lambda r: int(r.get(f"{rnd}_{n}_{metric}", 0) == 1), axis=1).mean()
            for n in agents]

r1_crop = _per(debate_df, debate_agents_list, "r1", "crop_correct")
r3_crop = _per(debate_df, debate_agents_list, "r3", "crop_correct")
r1_dis  = _per(debate_df, debate_agents_list, "r1", "disease_correct")
r3_dis  = _per(debate_df, debate_agents_list, "r3", "disease_correct")
r1_both = _per(debate_df, debate_agents_list, "r1", "both_correct")
r3_both = _per(debate_df, debate_agents_list, "r3", "both_correct")

for ax, (r1v, r3v, cv, title) in zip(
    [axes[0,0], axes[0,1], axes[1,0]],
    [(r1_crop, r3_crop, cons_crop, "Crop Accuracy: R1 vs R3"),
     (r1_dis,  r3_dis,  cons_dis,  "Disease Accuracy: R1 vs R3"),
     (r1_both, r3_both, cons_both, "Combined Accuracy: R1 vs R3")]):
    ax.bar(x-w/2, r1v, w, label="R1 (pre-debate)",  color=colors_rnd[0])
    ax.bar(x+w/2, r3v, w, label="R3 (post-debate)", color=colors_rnd[1])
    ax.axhline(cv, color="red", ls="--", lw=1.5, label=f"Consensus ({cv:.2f})")
    ax.set_xticks(x); ax.set_xticklabels(debate_agents_list, rotation=15, fontsize=9)
    ax.set_ylim(0, 1.05); ax.set_title(title); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.4)

ax = axes[0,2]
inf_mat = np.zeros((len(debate_agents_list), len(debate_agents_list)))
for i2, inf_n in enumerate(debate_agents_list):
    for j2, chg_n in enumerate(debate_agents_list):
        if i2 != j2: inf_mat[i2,j2] = influence_matrix.get((inf_n, chg_n), 0)
sns.heatmap(inf_mat, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax,
            xticklabels=debate_agents_list, yticklabels=debate_agents_list,
            cbar_kws={"label": "Times influenced"})
ax.set_xlabel("Agent CHANGED"); ax.set_ylabel("PERSUADER")
ax.set_title("Peer Influence Matrix"); ax.tick_params(rotation=20)

ax = axes[1,1]
stub_v = [stubbornness[n]/n_img for n in debate_agents_list]
flip_v = [flip_flop[n]/n_img    for n in debate_agents_list]
r2ch_v = [debate_df.apply(lambda r: r.get(f"r2_{n}_disease_changed", 0), axis=1).mean()
          for n in debate_agents_list]
x4 = np.arange(len(debate_agents_list)); w4 = 0.22
ax.bar(x4-w4, stub_v, w4, label="Stubborn",      color="#264653")
ax.bar(x4,    r2ch_v, w4, label="R1->R2 change", color=colors_rnd[0])
ax.bar(x4+w4, flip_v, w4, label="Flip-flop",     color="#e76f51")
ax.set_xticks(x4); ax.set_xticklabels(debate_agents_list, rotation=15, fontsize=9)
ax.set_ylim(0, 1.05); ax.set_title("Agent Behavior"); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.4)

ax = axes[1,2]
helped_v = [change_helped[n] for n in debate_agents_list]
hurt_v   = [change_hurt[n]   for n in debate_agents_list]
ax.bar(x-w/2, helped_v, w, label="Change Helped", color="#2a9d8f")
ax.bar(x+w/2, hurt_v,   w, label="Change Hurt",   color="#e63946")
ax.set_xticks(x); ax.set_xticklabels(debate_agents_list, rotation=15, fontsize=9)
ax.set_ylabel("Count"); ax.set_title("Impact of Changing Disease Verdict")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.4)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "panda_debate_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Pre/post debate viz ───────────────────────────────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 6))
fig2.suptitle(
    f"Consensus: Pre-Debate (R1 weighted) vs Post-Debate (R3 weighted)\n"
    f"Agents: {debate_agents_list}",
    fontsize=11, fontweight="bold", y=1.03)
metrics_3 = ["Crop Accuracy", "Disease Accuracy", "Combined Accuracy"]
pre_v  = [base_crop, base_dis, base_both]
post_v = [cons_crop, cons_dis, cons_both]
deltas = [p - r for p, r in zip(post_v, pre_v)]
xb = np.arange(3); wb = 0.34

ax = axes2[0]
bp = ax.bar(xb-wb/2, pre_v,  wb, label="Pre-debate",  color="#8ECAE6", edgecolor="white")
bq = ax.bar(xb+wb/2, post_v, wb, label="Post-debate", color="#023047", edgecolor="white")
for bar, v in zip(bp, pre_v):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.008, f"{v:.3f}",
            ha="center", fontsize=9, color="#023047", fontweight="bold")
for bar, v in zip(bq, post_v):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.008, f"{v:.3f}",
            ha="center", fontsize=9, color="white", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.1", facecolor="#023047", edgecolor="none"))
ax.set_xticks(xb); ax.set_xticklabels(metrics_3, fontsize=10)
ax.set_ylim(0, 1.05); ax.set_title("Consensus Before vs After Debate")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)

ax = axes2[1]
dc = ["#2a9d8f" if d>0.005 else "#e63946" if d<-0.005 else "#aaa" for d in deltas]
dbars = ax.bar(xb, deltas, color=dc, width=0.5)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
for bar, d in zip(dbars, deltas):
    arr = "^" if d > 0.005 else ("v" if d < -0.005 else "=")
    ax.text(bar.get_x()+bar.get_width()/2, d+(0.003 if d >= 0 else -0.008),
            f"{arr} {d:+.3f}", ha="center", fontsize=11, fontweight="bold",
            color="#2a9d8f" if d > 0.005 else "#e63946" if d < -0.005 else "#555")
ax.set_xticks(xb); ax.set_xticklabels(metrics_3, fontsize=10)
ax.set_ylabel("Delta (post - pre)"); ax.set_title("Debate Impact"); ax.grid(axis="y", alpha=0.3)

ax = axes2[2]
imp_v = [((debate_df["consensus_crop_correct"]==1)&(debate_df["baseline_crop_correct"]==0)).mean(),
         ((debate_df["consensus_disease_correct"]==1)&(debate_df["baseline_disease_correct"]==0)).mean(),
         ((debate_df["consensus_both_correct"]==1)&(debate_df["baseline_both_correct"]==0)).mean()]
deg_v = [((debate_df["consensus_crop_correct"]==0)&(debate_df["baseline_crop_correct"]==1)).mean(),
         ((debate_df["consensus_disease_correct"]==0)&(debate_df["baseline_disease_correct"]==1)).mean(),
         ((debate_df["consensus_both_correct"]==0)&(debate_df["baseline_both_correct"]==1)).mean()]
unch_v = [1-imp_v[j]-deg_v[j] for j in range(3)]
ax.bar(xb, imp_v,  0.5, label="Debate improved", color="#2a9d8f")
ax.bar(xb, unch_v, 0.5, bottom=imp_v, label="No change", color="#ddd")
ax.bar(xb, deg_v,  0.5, bottom=[imp_v[j]+unch_v[j] for j in range(3)],
       label="Debate hurt", color="#e63946")
ax.set_xticks(xb); ax.set_xticklabels(metrics_3, fontsize=10)
ax.set_ylim(0, 1.05); ax.set_title("Per-Image Debate Effect")
ax.legend(fontsize=8, loc="upper right"); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "baseline_vs_postdebate_consensus.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ All visualizations saved")

## Cell 19: Summary & Output Files

In [ ]:
print("=" * 65)
print("  VIDA + PANDA — CDDM SUMMARY")
print("=" * 65)
print(f"  Dataset      : CDDM — {len(pilot)} images")
print(f"  VIDA agents  : {agent_names}")
print(f"  PANDA agents : {debate_agents_list}")
print()
print("  VIDA RANKINGS:")
for rank, agent in enumerate(metrics_df.index, 1):
    ca = metrics_df.loc[agent, "Combined Acc"]
    rq = metrics_df.loc[agent, "Reasoning Quality"]
    print(f"  #{rank} {agent:<22} Combined={ca:.3f}  RQ={rq:.1f}/10")
print()
print(f"  PANDA: pre={base_both:.3f} -> post={cons_both:.3f}  delta={delta:+.3f}")
top_p = max(debate_agents_list,
            key=lambda n: sum(influence_matrix.get((n, o), 0)
                              for o in debate_agents_list if o != n))
print(f"  Top persuader: {top_p}")
print()
print("  OUTPUT FILES:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"    {p.relative_to(OUTPUT_DIR)} ({p.stat().st_size/1024:.1f} KB)")
print("=" * 65)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PANDA SAMPLE VISUALIZATION — Clean debate flow layout, research-style
# Layout per image: [photo] [R1 cards] [R2 cards] [R3 + consensus]
# Wrong → Right = gold, Right → Wrong = black
# ══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path
import numpy as np
import pandas as pd
import textwrap

# ── Config ───────────────────────────────────────────────────────────────────
short_d = {
    "GPT-4.1":        "GPT-4.1",
    "GPT-4.1-mini":   "GPT-mini",
    "Claude-Sonnet":  "C-Sonnet",
    "Claude-Haiku":   "C-Haiku",
    "Grok-4.20-Fast": "Grok-4.20",
    "Gemma-3n-E4B":   "Gemma-3n",
}

def _safe(v, fallback="?"):
    if pd.isna(v) or not str(v).strip():
        return fallback
    return str(v)

def _wrap(txt, width=48, max_lines=2):
    txt = _safe(txt, "")
    lines = textwrap.wrap(txt, width=width)
    if not lines:
        return "—"
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1].rstrip(".") + "..."
    return "\n".join(lines)

def _get_round_prediction(row, agent, rnd_key):
    return {
        "crop": row.get(f"{rnd_key}_{agent}_crop"),
        "disease": row.get(f"{rnd_key}_{agent}_disease"),
        "confidence": row.get(f"{rnd_key}_{agent}_confidence"),
        "influenced_by": row.get(f"{rnd_key}_{agent}_influenced_by"),
        "new_evidence": row.get(f"{rnd_key}_{agent}_new_evidence"),
    }

def _both_correct(p, gt_crop, gt_disease):
    crop_c = int(_safe(p.get("crop"), "") == gt_crop)
    dis_c  = int(_safe(p.get("disease"), "") == gt_disease)
    return crop_c, dis_c, int(crop_c and dis_c)

def _card_style(prev_ok, cur_ok, crop_c, dis_c, changed=False):
    # returns bg, accent, label
    if (prev_ok is not None) and (not prev_ok) and cur_ok:
        return "#fff7d6", "#9a6a00", "WRONG → RIGHT"
    if (prev_ok is not None) and prev_ok and (not cur_ok):
        return "#eeeeee", "#111111", "RIGHT → WRONG"
    if crop_c and dis_c:
        return "#e8f5e9", "#2e7d32", "CORRECT"
    if crop_c:
        return "#fff8e1", "#e65100", "CROP ONLY"
    if changed:
        return "#fff3e0", "#bf360c", "CHANGED"
    return "#ffebee", "#c62828", "WRONG"

def _rnd_ok_from_row(row, agent, rnd_key):
    return int(row.get(f"{rnd_key}_{agent}_both_correct", 0))

def _rnd_changed_from_row(row, agent, rnd_key):
    if rnd_key == "r1":
        return 0
    return int(row.get(f"{rnd_key}_{agent}_crop_changed", 0)) or int(row.get(f"{rnd_key}_{agent}_disease_changed", 0))

def _reasoning_text(row, agent, rnd_key):
    for k in [
        f"{rnd_key}_{agent}_reasoning",
        f"{rnd_key}_{agent}_new_evidence",
        f"{rnd_key}_{agent}_evidence",
        f"{rnd_key}_{agent}_rationale",
    ]:
        if k in row and pd.notna(row.get(k)) and str(row.get(k)).strip():
            return str(row.get(k))
    if rnd_key == "r1":
        return "Independent pre-debate diagnosis."
    return _safe(row.get(f"{rnd_key}_{agent}_new_evidence"), "No new evidence reported.")

# ── Build sample list if panda_samples does not exist ────────────────────────
if "panda_samples" not in globals():
    _agents_tmp = debate_agents_list
    def _n_final_correct(img_name):
        rr = debate_df[debate_df["image_name"] == img_name].iloc[0]
        return int(sum(rr.get(f"r3_{n}_both_correct", 0) for n in _agents_tmp))

    unique_imgs = debate_df["image_name"].unique().tolist()
    all_correct_s = [i for i in unique_imgs if _n_final_correct(i) == len(_agents_tmp)][:3]
    all_wrong_s   = [i for i in unique_imgs if _n_final_correct(i) == 0][:3]
    split_s       = [i for i in unique_imgs if 0 < _n_final_correct(i) < len(_agents_tmp)][:2]
    used          = set(all_correct_s + all_wrong_s + split_s)
    random_s      = [i for i in unique_imgs if i not in used][:2]
    panda_samples = all_correct_s + all_wrong_s + split_s + random_s

n_imgs   = len(panda_samples)
n_agents = len(debate_agents_list)

fig = plt.figure(figsize=(24, 4.75 * n_imgs))
fig.patch.set_facecolor("#f8f9fa")

outer = fig.add_gridspec(
    n_imgs, 4,
    wspace=0.045,
    hspace=0.075,
    width_ratios=[1.12, 1.18, 1.18, 1.30]
)

for row_i, img_name in enumerate(panda_samples):
    drow = debate_df[debate_df["image_name"] == img_name]
    if len(drow) == 0:
        continue

    drow = drow.iloc[0]

    gt_crop    = drow["gt_crop"]
    gt_disease = drow["gt_disease"]

    base_ok = bool(drow.get("baseline_both_correct", 0))
    cons_ok = bool(drow.get("consensus_both_correct", 0))

    cons_crop = _safe(drow.get("consensus_crop"), "?")
    cons_dis  = _safe(drow.get("consensus_disease"), "?")

    if cons_ok and not base_ok:
        impact_label = "DEBATE HELPED"
        impact_color = "#2e7d32"
    elif (not cons_ok) and base_ok:
        impact_label = "DEBATE HURT"
        impact_color = "#111111"
    elif cons_ok and base_ok:
        impact_label = "BOTH CORRECT"
        impact_color = "#2e7d32"
    else:
        impact_label = "BOTH WRONG"
        impact_color = "#c62828"

    # ── Col 0: Image ─────────────────────────────────────────────────────────
    ax_img = fig.add_subplot(outer[row_i, 0])

    img_path = next((p["image_path"] for p in pilot if p["image_path"].name == img_name), None)

    if img_path and Path(img_path).exists():
        ax_img.imshow(np.array(Image.open(img_path).convert("RGB")))
    else:
        ax_img.set_facecolor("#eeeeee")
        ax_img.text(
            0.5, 0.5,
            "Image not found",
            ha="center", va="center",
            transform=ax_img.transAxes,
            fontsize=9,
            color="#999"
        )

    ax_img.set_xticks([])
    ax_img.set_yticks([])

    for sp in ax_img.spines.values():
        sp.set_edgecolor(impact_color)
        sp.set_linewidth(3)

    ax_img.set_title(
        f"[{row_i + 1:02d}] {impact_label}",
        fontsize=10,
        fontweight="bold",
        color=impact_color,
        pad=5
    )

    ax_img.set_xlabel(
        f"GT: {gt_crop} / {gt_disease}",
        fontsize=8.5,
        labelpad=5,
        color="#333"
    )

    # ── Round columns ────────────────────────────────────────────────────────
    rnd_data = [
        ("R1  Pre-debate", "r1"),
        ("R2  Named debate", "r2"),
        ("R3  Final verdict", "r3"),
    ]

    for col_i, (rnd_label, rnd_key) in enumerate(rnd_data):
        extra_consensus = 1 if rnd_key == "r3" else 0

        inner = outer[row_i, col_i + 1].subgridspec(
            n_agents + extra_consensus,
            1,
            hspace=0.095,
            height_ratios=[1] * n_agents + ([0.82] if extra_consensus else [])
        )

        for ag_i, agent in enumerate(debate_agents_list):
            ax = fig.add_subplot(inner[ag_i])
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)

            p = _get_round_prediction(drow, agent, rnd_key)
            crop_c, dis_c, cur_ok = _both_correct(p, gt_crop, gt_disease)

            prev_ok = None
            if rnd_key == "r2":
                prev_ok = _rnd_ok_from_row(drow, agent, "r1")
            elif rnd_key == "r3":
                prev_ok = _rnd_ok_from_row(drow, agent, "r2")

            changed = _rnd_changed_from_row(drow, agent, rnd_key)
            bg, accent, status = _card_style(prev_ok, cur_ok, crop_c, dis_c, changed=changed)

            ax.set_facecolor(bg)

            for sp in ax.spines.values():
                sp.set_edgecolor(accent)
                sp.set_linewidth(1.35)

            header_h = 0.26
            header_color = accent if status == "RIGHT → WRONG" else bg

            ax.add_patch(
                plt.Rectangle(
                    (0, 1 - header_h),
                    1,
                    header_h,
                    transform=ax.transAxes,
                    facecolor=header_color,
                    edgecolor="none",
                    zorder=0
                )
            )

            header_text = "white" if status == "RIGHT → WRONG" else accent

            verdict_symbol = "✓" if cur_ok else ("~" if crop_c else "×")

            ax.text(
                0.025, 0.95,
                f"{verdict_symbol} {short_d.get(agent, agent)}",
                ha="left", va="top",
                transform=ax.transAxes,
                fontsize=8.3,
                fontweight="bold",
                color=header_text
            )

            ax.text(
                0.975, 0.95,
                status,
                ha="right", va="top",
                transform=ax.transAxes,
                fontsize=7.5,
                fontweight="bold",
                color=header_text
            )

            pred_crop = _safe(p.get("crop"), "?")
            pred_dis  = _safe(p.get("disease"), "?")

            ax.text(
                0.025, 0.675,
                "Prediction",
                ha="left", va="top",
                transform=ax.transAxes,
                fontsize=6.6,
                color="#666",
                fontweight="bold",
                style="italic"
            )

            ax.text(
                0.025, 0.555,
                _wrap(f"{pred_crop} / {pred_dis}", width=43, max_lines=2),
                ha="left", va="top",
                transform=ax.transAxes,
                fontsize=7.05,
                color="#222",
                linespacing=1.12
            )

            conf = p.get("confidence")
            conf_str = f"conf={conf:.2f}" if pd.notna(conf) and conf not in ("", None) else ""

            inf = _safe(p.get("influenced_by"), "")
            inf_str = ""
            if rnd_key != "r1" and inf not in ("", "None", "NONE", "?"):
                inf_str = f" ← {_wrap(inf, width=18, max_lines=1)}"

            meta = f"{conf_str}{inf_str}".strip()
            if not meta:
                meta = "independent" if rnd_key == "r1" else "no influence reported"

            ax.text(
                0.025, 0.335,
                meta,
                ha="left", va="top",
                transform=ax.transAxes,
                fontsize=6.45,
                color="#555"
            )

            reasoning = _reasoning_text(drow, agent, rnd_key)

            ax.text(
                0.025, 0.215,
                "Reasoning:",
                ha="left", va="top",
                transform=ax.transAxes,
                fontsize=6.3,
                color="#666",
                fontweight="bold",
                style="italic"
            )

            ax.text(
                0.19, 0.215,
                _wrap(reasoning, width=48, max_lines=2),
                ha="left", va="top",
                transform=ax.transAxes,
                fontsize=6.25,
                color="#333",
                linespacing=1.12
            )

            if row_i == 0 and ag_i == 0:
                ax.set_title(
                    rnd_label,
                    fontsize=10,
                    fontweight="bold",
                    color="#333",
                    pad=5
                )

        # Consensus box under R3
        if rnd_key == "r3":
            ax_cons = fig.add_subplot(inner[n_agents])
            ax_cons.set_xticks([])
            ax_cons.set_yticks([])
            ax_cons.set_xlim(0, 1)
            ax_cons.set_ylim(0, 1)

            cons_bg = "#c8e6c9" if cons_ok else "#eeeeee" if base_ok else "#ffcdd2"
            cons_ac = "#2e7d32" if cons_ok else "#111111" if base_ok else "#c62828"

            ax_cons.set_facecolor(cons_bg)

            for sp in ax_cons.spines.values():
                sp.set_edgecolor(cons_ac)
                sp.set_linewidth(2.2)

            ax_cons.text(
                0.5, 0.72,
                f"{'✓' if cons_ok else '×'} CONSENSUS",
                ha="center", va="center",
                transform=ax_cons.transAxes,
                fontsize=9.2,
                fontweight="bold",
                color=cons_ac
            )

            ax_cons.text(
                0.5, 0.43,
                _wrap(f"{cons_crop} / {cons_dis}", width=42, max_lines=2),
                ha="center", va="center",
                transform=ax_cons.transAxes,
                fontsize=7.2,
                color="#222",
                linespacing=1.15
            )

            ax_cons.text(
                0.5, 0.14,
                f"Baseline: {'correct' if base_ok else 'wrong'}",
                ha="center", va="center",
                transform=ax_cons.transAxes,
                fontsize=6.7,
                color="#555"
            )

fig.suptitle(
    "PANDA Block — Debate Sample Outputs | Clean Round-by-Round Deliberation Trace",
    fontsize=14,
    fontweight="bold",
    y=1.012,
    color="#111"
)

legend_patches = [
    mpatches.Patch(facecolor="#e8f5e9", edgecolor="#2e7d32", label="Correct"),
    mpatches.Patch(facecolor="#fff8e1", edgecolor="#e65100", label="Crop only"),
    mpatches.Patch(facecolor="#ffebee", edgecolor="#c62828", label="Wrong"),
    mpatches.Patch(facecolor="#fff7d6", edgecolor="#9a6a00", label="Wrong → Right"),
    mpatches.Patch(facecolor="#eeeeee", edgecolor="#111111", label="Right → Wrong"),
]

fig.legend(
    handles=legend_patches,
    loc="lower center",
    ncol=5,
    fontsize=9.5,
    bbox_to_anchor=(0.5, -0.01),
    framealpha=0.95,
    edgecolor="#cccccc"
)

plt.savefig(
    OUTPUT_DIR / "panda_sample_outputs_clean_flow.png",
    dpi=160,
    bbox_inches="tight",
    facecolor="#f8f9fa"
)

plt.show()

print("[OK] PANDA sample visualization saved: panda_sample_outputs_clean_flow.png")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PANDA SAMPLE VISUALIZATION — Auto-sized cards so full actual outputs fit
# Layout per image: [photo] [R1 cards] [R2 cards] [R3 cards + consensus]
#
# Key fix:
#   Card heights are computed dynamically from the wrapped raw model output.
#   This prevents the text from overflowing/overlapping.
# ══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path
import numpy as np
import pandas as pd
import textwrap
import re
import math

# ── Config ───────────────────────────────────────────────────────────────────
short_d = {
    "GPT-4.1":        "GPT-4.1",
    "GPT-4.1-mini":   "GPT-mini",
    "Claude-Sonnet":  "C-Sonnet",
    "Claude-Haiku":   "C-Haiku",
    "Grok-4.20-Fast": "Grok-4.20",
    "Gemma-3n-E4B":   "Gemma-3n",
}

rounds = ["r1", "r2", "r3"]

round_titles = {
    "r1": "R1 Pre-debate",
    "r2": "R2 Named debate",
    "r3": "R3 Final verdict",
}

# Wider figure = fewer wrapped lines = cleaner cards
FIG_WIDTH = 34

# Text settings
OUTPUT_WRAP_WIDTH = 105
PRED_WRAP_WIDTH   = 74
META_WRAP_WIDTH   = 92

FONT_HEADER = 9.2
FONT_STATUS = 8.0
FONT_LABEL  = 7.3
FONT_BODY   = 7.0
FONT_OUTPUT = 6.75

# Height model
# Increase LINE_HEIGHT_IN if text still feels tight.
BASE_CARD_HEIGHT_IN = 1.45
LINE_HEIGHT_IN      = 0.145
MIN_CARD_HEIGHT_IN  = 2.30
CONS_HEIGHT_IN      = 1.05
IMAGE_MIN_HEIGHT_IN = 6.8

# ── Helpers ──────────────────────────────────────────────────────────────────
def _safe(x, fallback="?"):
    if pd.isna(x) or not str(x).strip():
        return fallback
    return str(x).strip()

def _clean_text(txt):
    txt = _safe(txt, "")
    txt = txt.replace("\n", " ")
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

def _wrap_lines(txt, width=80):
    txt = _clean_text(txt)
    if not txt:
        return ["—"]
    return textwrap.wrap(
        txt,
        width=width,
        break_long_words=False,
        replace_whitespace=True,
    ) or ["—"]

def _wrap(txt, width=80, max_lines=None):
    lines = _wrap_lines(txt, width=width)
    if max_lines is not None and len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1].rstrip(".") + "..."
    return "\n".join(lines)

def _actual_output(img_name, agent, rnd):
    if rnd == "r1":
        return r1_raw_bank.get(img_name, {}).get(agent, "")
    return debate_transcripts.get(img_name, {}).get(rnd, {}).get(agent, "")

def _get_round_prediction(row, agent, rnd):
    return {
        "crop": row.get(f"{rnd}_{agent}_crop"),
        "disease": row.get(f"{rnd}_{agent}_disease"),
        "confidence": row.get(f"{rnd}_{agent}_confidence"),
        "influenced_by": row.get(f"{rnd}_{agent}_influenced_by"),
        "new_evidence": row.get(f"{rnd}_{agent}_new_evidence"),
    }

def _correct(pred, gt_crop, gt_disease):
    crop_c = int(_safe(pred.get("crop"), "") == gt_crop)
    dis_c  = int(_safe(pred.get("disease"), "") == gt_disease)
    both   = int(crop_c and dis_c)
    return crop_c, dis_c, both

def _round_ok(row, agent, rnd):
    return int(row.get(f"{rnd}_{agent}_both_correct", 0))

def _round_changed(row, agent, rnd):
    if rnd == "r1":
        return 0
    return int(row.get(f"{rnd}_{agent}_crop_changed", 0)) or int(row.get(f"{rnd}_{agent}_disease_changed", 0))

def _style(prev_ok, cur_ok, crop_c, dis_c, changed=False):
    # bg, accent, header, label
    if prev_ok is not None and (not prev_ok) and cur_ok:
        return "#fff7d6", "#9a6a00", "#f3cf66", "WRONG → RIGHT"

    if prev_ok is not None and prev_ok and (not cur_ok):
        return "#eeeeee", "#111111", "#111111", "RIGHT → WRONG"

    if crop_c and dis_c:
        return "#e8f5e9", "#2e7d32", "#b7dfb8", "CORRECT"

    if crop_c:
        return "#fff8e1", "#e65100", "#ffd180", "CROP ONLY"

    if changed:
        return "#fff3e0", "#bf360c", "#ffb74d", "CHANGED"

    return "#ffebee", "#c62828", "#ef9a9a", "WRONG"

def _ensure_panda_samples():
    if "panda_samples" in globals() and len(panda_samples) > 0:
        return panda_samples

    unique_imgs = debate_df["image_name"].unique().tolist()

    def _n_final_correct(img_name):
        rr = debate_df[debate_df["image_name"] == img_name].iloc[0]
        return int(sum(rr.get(f"r3_{n}_both_correct", 0) for n in debate_agents_list))

    all_correct_s = [i for i in unique_imgs if _n_final_correct(i) == len(debate_agents_list)][:3]
    all_wrong_s   = [i for i in unique_imgs if _n_final_correct(i) == 0][:3]
    split_s       = [i for i in unique_imgs if 0 < _n_final_correct(i) < len(debate_agents_list)][:2]

    used = set(all_correct_s + all_wrong_s + split_s)
    random_s = [i for i in unique_imgs if i not in used][:2]

    return all_correct_s + all_wrong_s + split_s + random_s

def _meta_text(pred, rnd):
    conf = pred.get("confidence")
    conf_str = f"{conf:.2f}" if pd.notna(conf) and str(conf).strip() else "—"

    if rnd == "r1":
        return "Mode: independent pre-debate answer"

    inf = _safe(pred.get("influenced_by"), "None")
    if inf in ["", "?", "None", "NONE", "none"]:
        return f"Confidence: {conf_str} | Influence: none reported"
    return f"Confidence: {conf_str} | Influence: {inf}"

def _estimate_card_height(row, img_name, agent, rnd):
    pred = _get_round_prediction(row, agent, rnd)

    pred_text = f"{_safe(pred.get('crop'), '?')} / {_safe(pred.get('disease'), '?')}"
    meta_text = _meta_text(pred, rnd)
    raw_out   = _actual_output(img_name, agent, rnd)

    n_pred_lines = len(_wrap_lines(pred_text, width=PRED_WRAP_WIDTH))
    n_meta_lines = len(_wrap_lines(meta_text, width=META_WRAP_WIDTH))
    n_out_lines  = len(_wrap_lines(raw_out, width=OUTPUT_WRAP_WIDTH))

    total_lines = n_pred_lines + n_meta_lines + n_out_lines

    return max(
        MIN_CARD_HEIGHT_IN,
        BASE_CARD_HEIGHT_IN + LINE_HEIGHT_IN * total_lines
    )

panda_samples = _ensure_panda_samples()

# ── Precompute dynamic heights ───────────────────────────────────────────────
image_specs = []

for img_name in panda_samples:
    drow = debate_df[debate_df["image_name"] == img_name]
    if len(drow) == 0:
        continue

    row = drow.iloc[0]

    # Make each agent row tall enough for the largest of R1/R2/R3 for that agent.
    agent_heights = []
    for agent in debate_agents_list:
        h = max(_estimate_card_height(row, img_name, agent, rnd) for rnd in rounds)
        agent_heights.append(h)

    block_height = max(
        IMAGE_MIN_HEIGHT_IN,
        sum(agent_heights) + CONS_HEIGHT_IN + 0.35
    )

    image_specs.append({
        "img_name": img_name,
        "row": row,
        "agent_heights": agent_heights,
        "block_height": block_height,
    })

total_fig_height = sum(spec["block_height"] for spec in image_specs)

# ── Figure ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(FIG_WIDTH, total_fig_height))
fig.patch.set_facecolor("#f8f9fa")

outer = fig.add_gridspec(
    len(image_specs),
    4,
    width_ratios=[1.10, 1.55, 1.55, 1.70],
    height_ratios=[spec["block_height"] for spec in image_specs],
    wspace=0.035,
    hspace=0.12
)

for row_i, spec in enumerate(image_specs):
    img_name = spec["img_name"]
    row = spec["row"]
    agent_heights = spec["agent_heights"]

    gt_crop = row["gt_crop"]
    gt_dis  = row["gt_disease"]

    cons_ok   = bool(row.get("consensus_both_correct", 0))
    base_ok   = bool(row.get("baseline_both_correct", 0))
    cons_crop = _safe(row.get("consensus_crop"), "?")
    cons_dis  = _safe(row.get("consensus_disease"), "?")

    if cons_ok and not base_ok:
        impact = "DEBATE HELPED"
        impact_color = "#2e7d32"
    elif (not cons_ok) and base_ok:
        impact = "DEBATE HURT"
        impact_color = "#111111"
    elif cons_ok and base_ok:
        impact = "BOTH CORRECT"
        impact_color = "#2e7d32"
    else:
        impact = "BOTH WRONG"
        impact_color = "#c62828"

    # ── Image panel ─────────────────────────────────────────────────────────
    ax_img = fig.add_subplot(outer[row_i, 0])

    img_path = next((p["image_path"] for p in pilot if p["image_path"].name == img_name), None)

    if img_path and Path(img_path).exists():
        ax_img.imshow(np.array(Image.open(img_path).convert("RGB")))
    else:
        ax_img.set_facecolor("#eeeeee")
        ax_img.text(
            0.5, 0.5,
            "Image not found",
            ha="center", va="center",
            transform=ax_img.transAxes,
            fontsize=10,
            color="#999"
        )

    ax_img.set_xticks([])
    ax_img.set_yticks([])

    for sp in ax_img.spines.values():
        sp.set_edgecolor(impact_color)
        sp.set_linewidth(3)

    ax_img.set_title(
        f"[{row_i + 1:02d}] {impact}",
        fontsize=11,
        fontweight="bold",
        color=impact_color,
        pad=8
    )

    ax_img.set_xlabel(
        f"GT: {gt_crop} / {gt_dis}",
        fontsize=9.5,
        labelpad=8,
        color="#222"
    )

    # ── Round columns ────────────────────────────────────────────────────────
    for col_i, rnd in enumerate(rounds):
        has_consensus = rnd == "r3"

        height_ratios = agent_heights + ([CONS_HEIGHT_IN] if has_consensus else [])

        inner = outer[row_i, col_i + 1].subgridspec(
            len(height_ratios),
            1,
            hspace=0.10,
            height_ratios=height_ratios
        )

        for ag_i, agent in enumerate(debate_agents_list):
            ax = fig.add_subplot(inner[ag_i])

            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)

            pred = _get_round_prediction(row, agent, rnd)
            crop_c, dis_c, cur_ok = _correct(pred, gt_crop, gt_dis)

            prev_ok = None
            if rnd == "r2":
                prev_ok = _round_ok(row, agent, "r1")
            elif rnd == "r3":
                prev_ok = _round_ok(row, agent, "r2")

            changed = _round_changed(row, agent, rnd)
            bg, accent, header_bg, status = _style(prev_ok, cur_ok, crop_c, dis_c, changed)

            ax.set_facecolor(bg)

            for sp in ax.spines.values():
                sp.set_edgecolor(accent)
                sp.set_linewidth(1.45)

            # Header band
            header_h = 0.145
            ax.add_patch(
                plt.Rectangle(
                    (0, 1 - header_h),
                    1,
                    header_h,
                    transform=ax.transAxes,
                    facecolor=header_bg,
                    edgecolor="none",
                    zorder=0
                )
            )

            header_text_color = "white" if status == "RIGHT → WRONG" else accent
            verdict_symbol = "✓" if cur_ok else ("~" if crop_c else "×")

            ax.text(
                0.020, 0.975,
                f"{verdict_symbol} {short_d.get(agent, agent)}",
                ha="left",
                va="top",
                transform=ax.transAxes,
                fontsize=FONT_HEADER,
                fontweight="bold",
                color=header_text_color
            )

            ax.text(
                0.980, 0.975,
                status,
                ha="right",
                va="top",
                transform=ax.transAxes,
                fontsize=FONT_STATUS,
                fontweight="bold",
                color=header_text_color
            )

            # Prediction
            pred_crop = _safe(pred.get("crop"), "?")
            pred_dis  = _safe(pred.get("disease"), "?")
            pred_text = _wrap(f"{pred_crop} / {pred_dis}", width=PRED_WRAP_WIDTH)

            ax.text(
                0.025, 0.815,
                "Prediction",
                ha="left",
                va="top",
                transform=ax.transAxes,
                fontsize=FONT_LABEL,
                color="#555",
                fontweight="bold",
                style="italic"
            )

            ax.text(
                0.025, 0.760,
                pred_text,
                ha="left",
                va="top",
                transform=ax.transAxes,
                fontsize=FONT_BODY,
                color="#111",
                linespacing=1.18
            )

            # Meta
            meta = _wrap(_meta_text(pred, rnd), width=META_WRAP_WIDTH)

            ax.text(
                0.025, 0.640,
                meta,
                ha="left",
                va="top",
                transform=ax.transAxes,
                fontsize=FONT_BODY,
                color="#444",
                linespacing=1.14
            )

            ax.axhline(0.585, color="#d0d0d0", linewidth=0.55)

            # Full actual output
            raw_out = _actual_output(img_name, agent, rnd)
            output_text = _wrap(raw_out, width=OUTPUT_WRAP_WIDTH, max_lines=None)

            ax.text(
                0.025, 0.550,
                "Actual model output",
                ha="left",
                va="top",
                transform=ax.transAxes,
                fontsize=FONT_LABEL,
                color="#555",
                fontweight="bold",
                style="italic"
            )

            ax.text(
                0.025, 0.490,
                output_text,
                ha="left",
                va="top",
                transform=ax.transAxes,
                fontsize=FONT_OUTPUT,
                color="#222",
                linespacing=1.18
            )

            if row_i == 0 and ag_i == 0:
                ax.set_title(
                    round_titles[rnd],
                    fontsize=12.5,
                    fontweight="bold",
                    color="#222",
                    pad=10
                )

        # Consensus card below R3
        if has_consensus:
            axc = fig.add_subplot(inner[len(debate_agents_list)])

            axc.set_xticks([])
            axc.set_yticks([])
            axc.set_xlim(0, 1)
            axc.set_ylim(0, 1)

            cons_bg = "#c8e6c9" if cons_ok else "#ffcdd2"
            cons_ac = "#2e7d32" if cons_ok else "#c62828"

            if (not cons_ok) and base_ok:
                cons_bg = "#eeeeee"
                cons_ac = "#111111"

            axc.set_facecolor(cons_bg)

            for sp in axc.spines.values():
                sp.set_edgecolor(cons_ac)
                sp.set_linewidth(2.2)

            axc.text(
                0.5, 0.68,
                f"{'✓' if cons_ok else '×'} CONSENSUS",
                ha="center",
                va="center",
                transform=axc.transAxes,
                fontsize=10,
                fontweight="bold",
                color=cons_ac
            )

            axc.text(
                0.5, 0.39,
                _wrap(f"{cons_crop} / {cons_dis}", width=65),
                ha="center",
                va="center",
                transform=axc.transAxes,
                fontsize=8.2,
                color="#222",
                linespacing=1.18
            )

            axc.text(
                0.5, 0.13,
                f"Baseline consensus: {'correct' if base_ok else 'wrong'}",
                ha="center",
                va="center",
                transform=axc.transAxes,
                fontsize=7.3,
                color="#444"
            )

# ── Title and legend ─────────────────────────────────────────────────────────
fig.suptitle(
    "PANDA Block — Auto-Sized Debate Cards with Full Actual Model Outputs",
    fontsize=17,
    fontweight="bold",
    y=1.002,
    color="#111"
)

legend_patches = [
    mpatches.Patch(facecolor="#e8f5e9", edgecolor="#2e7d32", label="Correct"),
    mpatches.Patch(facecolor="#fff8e1", edgecolor="#e65100", label="Crop only"),
    mpatches.Patch(facecolor="#ffebee", edgecolor="#c62828", label="Wrong"),
    mpatches.Patch(facecolor="#fff7d6", edgecolor="#9a6a00", label="Wrong → Right"),
    mpatches.Patch(facecolor="#eeeeee", edgecolor="#111111", label="Right → Wrong"),
]

fig.legend(
    handles=legend_patches,
    loc="lower center",
    ncol=5,
    fontsize=10.5,
    bbox_to_anchor=(0.5, -0.004),
    framealpha=0.96,
    edgecolor="#cccccc"
)

plt.savefig(
    OUTPUT_DIR / "panda_sample_outputs_auto_sized_full_text.png",
    dpi=170,
    bbox_inches="tight",
    facecolor="#f8f9fa"
)

plt.show()

print("[OK] PANDA auto-sized full-text visualization saved: panda_sample_outputs_auto_sized_full_text.png")